In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Rename the "Item" column values
items_to_keep = ['73292']
filtered_data.loc[~filtered_data['Item'].isin(items_to_keep), 'Item'] = 'Group'

# List of items to fit models to
categories = ['Group', '73292']

# Define model functions
def logistic_diffusion(t, K, r, t0):
    return K / (1 + np.exp(-r * (t - t0)))

def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

model_predictions = {}

# Fit models, select the best for each category, and predict out to 2035
for category in categories:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Create a time index for fitting models and extending predictions
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + (2035 - category_data.index[-1].year) * 12)

    models = {
        "Logistic Diffusion": logistic_diffusion,
        "Gompertz": gompertz,
        "Bass Diffusion": bass_diffusion
    }
    results = {}

    for model_name, model_func in models.items():
        try:
            params, _ = curve_fit(model_func, current_time_index, category_data, maxfev=10000)
            predicted = model_func(current_time_index, *params)
            future_predicted = model_func(future_time_index, *params)
            map = mean_absolute_percentage_error(category_data, predicted)
            results[model_name] = {'MAPE': map, 'params': params, 'future_predicted': future_predicted}
        except Exception as e:
            print(f"Error fitting {model_name} for '{category}': {e}")
            continue

    # Fit SARIMA model
    try:
        sarima_model = SARIMAX(category_data, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
        sarima_fit = sarima_model.fit(disp=False)
        sarima_future_pred = sarima_fit.get_forecast(steps=len(future_time_index)).predicted_mean
        sarima_mape = mean_absolute_percentage_error(category_data, sarima_fit.fittedvalues)
        results["SARIMA"] = {'MAPE': sarima_mape, 'future_predicted': sarima_future_pred}
    except Exception as e:
        print(f"Error fitting SARIMA for '{category}': {e}")

    # Select the best model
    if results:
        best_model = min(results, key=lambda x: results[x]['MAPE'])
        model_predictions[category] = results[best_model]['future_predicted']

# Calculate the intersection point between "Group" and "73292"
if 'Group' in model_predictions and '73292' in model_predictions:
    group_pred = model_predictions['Group']
    item_73292_pred = model_predictions['73292']
    intersection = np.where(np.diff(np.sign(group_pred - item_73292_pred)))[0]

    if intersection.size > 0:
        intersection_point = intersection[0]
        intersection_date = pd.date_range(
            start=category_data.index[-1] + pd.offsets.MonthEnd(1),
            periods=len(future_time_index),
            freq='M'
        )[intersection_point]

        print(f"The predicted intersection date is: {intersection_date.strftime('%Y-%m')}")

        # Plot the intersection
        plt.figure(figsize=(14, 7))
        plt.plot(pd.date_range(start=category_data.index[-1] + pd.offsets.MonthEnd(1), periods=len(future_time_index), freq='M'),
                 group_pred, label='Group Prediction', linestyle='--')
        plt.plot(pd.date_range(start=category_data.index[-1] + pd.offsets.MonthEnd(1), periods=len(future_time_index), freq='M'),
                 item_73292_pred, label='73292 Prediction', linestyle='--')
        plt.axvline(intersection_date, color='red', linestyle=':', label=f'Intersection: {intersection_date.strftime("%Y-%m")}')
        plt.title('Prediction Comparison and Intersection Point')
        plt.xlabel('Date')
        plt.ylabel('Value')
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No intersection found in the predicted range.")
        # Plot the predictions
        plt.figure(figsize=(14, 7))
        plt.plot(pd.date_range(start=category_data.index[-1] + pd.offsets.MonthEnd(1), periods=len(future_time_index), freq='M'),
                 group_pred, label='Group Prediction', linestyle='--')
        plt.plot(pd.date_range(start=category_data.index[-1] + pd.offsets.MonthEnd(1), periods=len(future_time_index), freq='M'),
                 item_73292_pred, label='73292 Prediction', linestyle='--')
        plt.title('Prediction Comparison')
        plt.xlabel('Date')
        plt.ylabel('Value')
        plt.legend()
        plt.tight_layout()
        plt.show()
else:
    print("Predictions for 'Group' or '73292' are not available.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Rename the "Item" column values
items_to_keep = ['73292']
group_1_items = ['73358', '73359']
filtered_data.loc[~filtered_data['Item'].isin(items_to_keep + group_1_items), 'Item'] = 'Other'
filtered_data.loc[filtered_data['Item'].isin(group_1_items), 'Item'] = 'Group 1'

# List of items to fit models to
categories = ['Group 1', '73292']

# Define model functions
def logistic_diffusion(t, K, r, t0):
    return K / (1 + np.exp(-r * (t - t0)))

def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

model_predictions = {}

# Fit models, select the best for each category, and predict out to 2035
for category in categories:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Create a time index for fitting models and extending predictions
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + (2035 - category_data.index[-1].year) * 12)

    models = {
        "Logistic Diffusion": logistic_diffusion,
        "Gompertz": gompertz,
        "Bass Diffusion": bass_diffusion
    }
    results = {}

    for model_name, model_func in models.items():
        try:
            params, _ = curve_fit(model_func, current_time_index, category_data, maxfev=10000)
            predicted = model_func(current_time_index, *params)
            future_predicted = model_func(future_time_index, *params)
            map = mean_absolute_percentage_error(category_data, predicted)
            results[model_name] = {'MAPE': map, 'params': params, 'future_predicted': future_predicted}
        except Exception as e:
            print(f"Error fitting {model_name} for '{category}': {e}")
            continue

    # Fit SARIMA model
    try:
        sarima_model = SARIMAX(category_data, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
        sarima_fit = sarima_model.fit(disp=False)
        sarima_future_pred = sarima_fit.get_forecast(steps=len(future_time_index)).predicted_mean
        sarima_mape = mean_absolute_percentage_error(category_data, sarima_fit.fittedvalues)
        results["SARIMA"] = {'MAPE': sarima_mape, 'future_predicted': sarima_future_pred}
    except Exception as e:
        print(f"Error fitting SARIMA for '{category}': {e}")

    # Select the best model
    if results:
        best_model = min(results, key=lambda x: results[x]['MAPE'])
        model_predictions[category] = results[best_model]['future_predicted']

# Calculate the intersection point between "Group 1" and "73292"
if 'Group 1' in model_predictions and '73292' in model_predictions:
    group_pred = model_predictions['Group 1']
    item_73292_pred = model_predictions['73292']
    intersection = np.where(np.diff(np.sign(group_pred - item_73292_pred)))[0]

    if intersection.size > 0:
        intersection_point = intersection[0]
        intersection_date = pd.date_range(
            start=category_data.index[-1] + pd.offsets.MonthEnd(1),
            periods=len(future_time_index),
            freq='M'
        )[intersection_point]

        print(f"The predicted intersection date is: {intersection_date.strftime('%Y-%m')}")

        # Plot the intersection
        plt.figure(figsize=(14, 7))
        plt.plot(pd.date_range(start=category_data.index[-1] + pd.offsets.MonthEnd(1), periods=len(future_time_index), freq='M'),
                 group_pred, label='Group 1 Prediction', linestyle='--')
        plt.plot(pd.date_range(start=category_data.index[-1] + pd.offsets.MonthEnd(1), periods=len(future_time_index), freq='M'),
                 item_73292_pred, label='73292 Prediction', linestyle='--')
        plt.axvline(intersection_date, color='red', linestyle=':', label=f'Intersection: {intersection_date.strftime("%Y-%m")}')
        plt.title('Prediction Comparison and Intersection Point')
        plt.xlabel('Date')
        plt.ylabel('Value')
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No intersection found in the predicted range.")
        # Plot the predictions
        plt.figure(figsize=(14, 7))
        plt.plot(pd.date_range(start=category_data.index[-1] + pd.offsets.MonthEnd(1), periods=len(future_time_index), freq='M'),
                 group_pred, label='Group 1 Prediction', linestyle='--')
        plt.plot(pd.date_range(start=category_data.index[-1] + pd.offsets.MonthEnd(1), periods=len(future_time_index), freq='M'),
                 item_73292_pred, label='73292 Prediction', linestyle='--')
        plt.title('Prediction Comparison')
        plt.xlabel('Date')
        plt.ylabel('Value')
        plt.legend()
        plt.tight_layout()
        plt.show()
else:
    print("Predictions for 'Group 1' or '73292' are not available.")


In [ ]:
if 'Group 1' in model_predictions:
    group_pred = model_predictions['Group 1']
    average_group_pred = np.mean(group_pred)
    print(f"Average predicted value for 'Group 1': {average_group_pred}")
else:
    print("Predictions for 'Group 1' are not available.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Rename the "Item" column values
items_to_keep = ['73292']
group_1_items = ['73358', '73359']
group_2_items = ['73298', '73358', '73359', '73401', '73402', '73422']
filtered_data.loc[~filtered_data['Item'].isin(items_to_keep + group_1_items + group_2_items), 'Item'] = 'Other'
filtered_data.loc[filtered_data['Item'].isin(group_1_items), 'Item'] = 'Group 1'
filtered_data.loc[filtered_data['Item'].isin(group_2_items), 'Item'] = 'Group 2'

# List of items to fit models to
categories = ['Group 1', 'Group 2', '73292']

# Define model functions
def logistic_diffusion(t, K, r, t0):
    return K / (1 + np.exp(-r * (t - t0)))

def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

model_predictions = {}

# Fit models, select the best for each category, and predict out to 2035
for category in categories:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Create a time index for fitting models and extending predictions
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + (2035 - category_data.index[-1].year) * 12)

    models = {
        "Logistic Diffusion": logistic_diffusion,
        "Gompertz": gompertz,
        "Bass Diffusion": bass_diffusion
    }
    results = {}

    for model_name, model_func in models.items():
        try:
            params, _ = curve_fit(model_func, current_time_index, category_data, maxfev=10000)
            predicted = model_func(current_time_index, *params)
            future_predicted = model_func(future_time_index, *params)
            map = mean_absolute_percentage_error(category_data, predicted)
            results[model_name] = {'MAPE': map, 'params': params, 'future_predicted': future_predicted}
        except Exception as e:
            print(f"Error fitting {model_name} for '{category}': {e}")
            continue

    # Fit SARIMA model
    try:
        sarima_model = SARIMAX(category_data, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
        sarima_fit = sarima_model.fit(disp=False)
        sarima_future_pred = sarima_fit.get_forecast(steps=len(future_time_index)).predicted_mean
        sarima_mape = mean_absolute_percentage_error(category_data, sarima_fit.fittedvalues)
        results["SARIMA"] = {'MAPE': sarima_mape, 'future_predicted': sarima_future_pred}
    except Exception as e:
        print(f"Error fitting SARIMA for '{category}': {e}")

    # Select the best model
    if results:
        best_model = min(results, key=lambda x: results[x]['MAPE'])
        model_predictions[category] = results[best_model]['future_predicted']

# Calculate the intersection point between "Group 1" and "73292"
if 'Group 1' in model_predictions and '73292' in model_predictions:
    group_1_pred = model_predictions['Group 1']
    item_73292_pred = model_predictions['73292']
    intersection = np.where(np.diff(np.sign(group_1_pred - item_73292_pred)))[0]

    if intersection.size > 0:
        intersection_point = intersection[0]
        intersection_date = pd.date_range(
            start=category_data.index[-1] + pd.offsets.MonthEnd(1),
            periods=len(future_time_index),
            freq='M'
        )[intersection_point]

        print(f"The predicted intersection date for Group 1 and 73292 is: {intersection_date.strftime('%Y-%m')}")

        # Plot the intersection
        plt.figure(figsize=(14, 7))
        plt.plot(pd.date_range(start=category_data.index[-1] + pd.offsets.MonthEnd(1), periods=len(future_time_index), freq='M'),
                 group_1_pred, label='Group 1 Prediction', linestyle='--')
        plt.plot(pd.date_range(start=category_data.index[-1] + pd.offsets.MonthEnd(1), periods=len(future_time_index), freq='M'),
                 item_73292_pred, label='73292 Prediction', linestyle='--')
        plt.axvline(intersection_date, color='red', linestyle=':', label=f'Intersection: {intersection_date.strftime("%Y-%m")}')
        plt.title('Prediction Comparison and Intersection Point (Group 1 vs 73292)')
        plt.xlabel('Date')
        plt.ylabel('Value')
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No intersection found in the predicted range for Group 1 and 73292.")

# Calculate the intersection point between "Group 2" and "73292"
if 'Group 2' in model_predictions and '73292' in model_predictions:
    group_2_pred = model_predictions['Group 2']
    item_73292_pred = model_predictions['73292']
    intersection = np.where(np.diff(np.sign(group_2_pred - item_73292_pred)))[0]

    if intersection.size > 0:
        intersection_point = intersection[0]
        intersection_date = pd.date_range(
            start=category_data.index[-1] + pd.offsets.MonthEnd(1),
            periods=len(future_time_index),
            freq='M'
        )[intersection_point]

        print(f"The predicted intersection date for Group 2 and 73292 is: {intersection_date.strftime('%Y-%m')}")

        # Plot the intersection
        plt.figure(figsize=(14, 7))
        plt.plot(pd.date_range(start=category_data.index[-1] + pd.offsets.MonthEnd(1), periods=len(future_time_index), freq='M'),
                 group_2_pred, label='Group 2 Prediction', linestyle='--')
        plt.plot(pd.date_range(start=category_data.index[-1] + pd.offsets.MonthEnd(1), periods=len(future_time_index), freq='M'),
                 item_73292_pred, label='73292 Prediction', linestyle='--')
        plt.axvline(intersection_date, color='red', linestyle=':', label=f'Intersection: {intersection_date.strftime("%Y-%m")}')
        plt.title('Prediction Comparison and Intersection Point (Group 2 vs 73292)')
        plt.xlabel('Date')
        plt.ylabel('Value')
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No intersection found in the predicted range for Group 2 and 73292.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Rename the "Item" column values
items_to_keep = ['73292']
group_1_items = ['73358', '73359']
group_2_items = ['73298', '73358', '73359', '73401', '73402', '73422']
filtered_data.loc[~filtered_data['Item'].isin(items_to_keep + group_1_items + group_2_items), 'Item'] = 'Other'
filtered_data.loc[filtered_data['Item'].isin(group_1_items), 'Item'] = 'Group 1'
filtered_data.loc[filtered_data['Item'].isin(group_2_items), 'Item'] = 'Group 2'

# List of items to fit models to
categories = ['Group 1', 'Group 2', '73292']

# Define model functions
def logistic_diffusion(t, K, r, t0):
    return K / (1 + np.exp(-r * (t - t0)))

def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

model_predictions = {}

# Fit models, select the best for each category, and predict out to 2035
for category in categories:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Create a time index for fitting models and extending predictions
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + (2035 - category_data.index[-1].year) * 12)

    if category == '73292':
        # Fit SARIMA model for 73292
        try:
            sarima_model = SARIMAX(category_data, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
            sarima_fit = sarima_model.fit(disp=False)
            sarima_future_pred = sarima_fit.get_forecast(steps=len(future_time_index)).predicted_mean
            model_predictions[category] = sarima_future_pred
        except Exception as e:
            print(f"Error fitting SARIMA for '{category}': {e}")
    else:
        # Fit Bass Diffusion model for Group 1 and Group 2
        try:
            params, _ = curve_fit(bass_diffusion, current_time_index, category_data, maxfev=10000)
            future_predicted = bass_diffusion(future_time_index, *params)
            model_predictions[category] = future_predicted
        except Exception as e:
            print(f"Error fitting Bass Diffusion for '{category}': {e}")

# Calculate the intersection point between "Group 1" and "73292"
if 'Group 1' in model_predictions and '73292' in model_predictions:
    group_1_pred = model_predictions['Group 1']
    item_73292_pred = model_predictions['73292']
    intersection = np.where(np.diff(np.sign(group_1_pred - item_73292_pred)))[0]

    if intersection.size > 0:
        intersection_point = intersection[0]
        intersection_date = pd.date_range(
            start=category_data.index[-1] + pd.offsets.MonthEnd(1),
            periods=len(future_time_index),
            freq='M'
        )[intersection_point]

        print(f"The predicted intersection date for Group 1 and 73292 is: {intersection_date.strftime('%Y-%m')}")

        # Plot the intersection
        plt.figure(figsize=(14, 7))
        plt.plot(pd.date_range(start=category_data.index[-1] + pd.offsets.MonthEnd(1), periods=len(future_time_index), freq='M'),
                 group_1_pred, label='Group 1 Prediction', linestyle='--')
        plt.plot(pd.date_range(start=category_data.index[-1] + pd.offsets.MonthEnd(1), periods=len(future_time_index), freq='M'),
                 item_73292_pred, label='73292 Prediction', linestyle='--')
        plt.axvline(intersection_date, color='red', linestyle=':', label=f'Intersection: {intersection_date.strftime("%Y-%m")}')
        plt.title('Prediction Comparison and Intersection Point (Group 1 vs 73292)')
        plt.xlabel('Date')
        plt.ylabel('Value')
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No intersection found in the predicted range for Group 1 and 73292.")

# Calculate the intersection point between "Group 2" and "73292"
if 'Group 2' in model_predictions and '73292' in model_predictions:
    group_2_pred = model_predictions['Group 2']
    item_73292_pred = model_predictions['73292']
    intersection = np.where(np.diff(np.sign(group_2_pred - item_73292_pred)))[0]

    if intersection.size > 0:
        intersection_point = intersection[0]
        intersection_date = pd.date_range(
            start=category_data.index[-1] + pd.offsets.MonthEnd(1),
            periods=len(future_time_index),
            freq='M'
        )[intersection_point]

        print(f"The predicted intersection date for Group 2 and 73292 is: {intersection_date.strftime('%Y-%m')}")

        # Plot the intersection
        plt.figure(figsize=(14, 7))
        plt.plot(pd.date_range(start=category_data.index[-1] + pd.offsets.MonthEnd(1), periods=len(future_time_index), freq='M'),
                 group_2_pred, label='Group 2 Prediction', linestyle='--')
        plt.plot(pd.date_range(start=category_data.index[-1] + pd.offsets.MonthEnd(1), periods=len(future_time_index), freq='M'),
                 item_73292_pred, label='73292 Prediction', linestyle='--')
        plt.axvline(intersection_date, color='red', linestyle=':', label=f'Intersection: {intersection_date.strftime("%Y-%m")}')
        plt.title('Prediction Comparison and Intersection Point (Group 2 vs 73292)')
        plt.xlabel('Date')
        plt.ylabel('Value')
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No intersection found in the predicted range for Group 2 and 73292.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Rename the "Item" column values
items_to_group = ['73358', '73359']
filtered_data.loc[filtered_data['Item'].isin(items_to_group), 'Item'] = 'Group 1'

# List of items to fit models to
categories = ['Group 1', '73292']

# Define model functions
def logistic_diffusion(t, K, r, t0):
    return K / (1 + np.exp(-r * (t - t0)))

def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

model_predictions = {}

# Fit models, select the best for each category, and predict out to 2035
for category in categories:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()
    
    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Create a time index for fitting models and extending predictions
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + (2035 - category_data.index[-1].year) * 12)

    models = {
        "Logistic Diffusion": logistic_diffusion,
        "Gompertz": gompertz,
        "Bass Diffusion": bass_diffusion
    }
    best_model = None
    results = {}

    for model_name, model_func in models.items():
        try:
            params, _ = curve_fit(model_func, current_time_index, category_data, maxfev=10000)
            predicted = model_func(current_time_index, *params)
            future_predicted = model_func(future_time_index, *params)
            mae = mean_absolute_error(category_data, predicted)
            results[model_name] = {'MAE': mae, 'params': params, 'future_predicted': future_predicted}
        except Exception as e:
            print(f"Error fitting {model_name} for '{category}': {e}")
            continue

    # Fit SARIMA model
    try:
        sarima_model = SARIMAX(category_data, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
        sarima_fit = sarima_model.fit(disp=False)
        sarima_future_pred = sarima_fit.get_forecast(steps=len(future_time_index)).predicted_mean
        sarima_mae = mean_absolute_error(category_data, sarima_fit.fittedvalues)
        results["SARIMA"] = {'MAE': sarima_mae, 'future_predicted': sarima_future_pred}
    except Exception as e:
        print(f"Error fitting SARIMA for '{category}': {e}")

    # Select the best model
    if results:
        best_model = min(results, key=lambda x: results[x]['MAE'])
        model_predictions[category] = results[best_model]['future_predicted']

# Calculate the intersection point between "Group 1" and "73292"
if 'Group 1' in model_predictions and '73292' in model_predictions:
    group_pred = model_predictions['Group 1']
    item_73292_pred = model_predictions['73292']
    intersection = np.where(np.diff(np.sign(group_pred - item_73292_pred)))[0]

    if intersection.size > 0:
        intersection_point = intersection[0]
        intersection_date = pd.date_range(
            start=category_data.index[-1] + pd.offsets.MonthEnd(1),
            periods=len(future_time_index),
            freq='M'
        )[intersection_point]

        print(f"The predicted intersection date is: {intersection_date.strftime('%Y-%m')}")

        # Plot the intersection
        plt.figure(figsize=(14, 7))
        plt.plot(pd.date_range(start=category_data.index[-1] + pd.offsets.MonthEnd(1), periods=len(future_time_index), freq='M'),
                 group_pred, label='Group 1 Prediction', linestyle='--')
        plt.plot(pd.date_range(start=category_data.index[-1] + pd.offsets.MonthEnd(1), periods=len(future_time_index), freq='M'),
                 item_73292_pred, label='73292 Prediction', linestyle='--')
        plt.axvline(intersection_date, color='red', linestyle=':', label=f'Intersection: {intersection_date.strftime("%Y-%m")}')
        plt.title('Prediction Comparison and Intersection Point')
        plt.xlabel('Date')
        plt.ylabel('Value')
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No intersection found in the predicted range.")
else:
    print("Predictions for 'Group 1' or '73292' are not available.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import mean_absolute_error

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Rename the "Item" column values explicitly including 73358 and 73359 in "Group 1"
items_to_keep = ['73292']
filtered_data.loc[~filtered_data['Item'].isin(items_to_keep + ['73358', '73359']), 'Item'] = 'Group 1'

# List of items to fit models to
categories = ['Group 1', '73292']

# Define model functions
def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

model_predictions = {}

# Fit the specified models and predict out to 2035
for category in categories:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()
    
    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Create a time index for fitting models and extending predictions
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + (2035 - category_data.index[-1].year) * 12)

    # Fit Gompertz curve for "73292" and Bass diffusion curve for "Group 1"
    if category == '73292':
        try:
            params, _ = curve_fit(gompertz, current_time_index, category_data, maxfev=10000)
            predicted = gompertz(current_time_index, *params)
            future_predicted = gompertz(future_time_index, *params)
            model_predictions[category] = future_predicted
            print(f"Gompertz model parameters for '73292': {params}")
        except Exception as e:
            print(f"Error fitting Gompertz model for '{category}': {e}")
            continue

    elif category == 'Group 1':
        try:
            params, _ = curve_fit(bass_diffusion, current_time_index, category_data, maxfev=10000)
            predicted = bass_diffusion(current_time_index, *params)
            future_predicted = bass_diffusion(future_time_index, *params)
            model_predictions[category] = future_predicted
            print(f"Bass diffusion model parameters for 'Group 1': {params}")
        except Exception as e:
            print(f"Error fitting Bass diffusion model for '{category}': {e}")
            continue

# Calculate the intersection point between "Group 1" and "73292"
if 'Group 1' in model_predictions and '73292' in model_predictions:
    group_pred = model_predictions['Group 1']
    item_73292_pred = model_predictions['73292']
    future_index = pd.date_range(
        start=category_data.index[-1] + pd.offsets.MonthEnd(1),
        periods=len(future_time_index),
        freq='M'
    )
    intersection = np.where(np.diff(np.sign(group_pred - item_73292_pred)))[0]

    if intersection.size > 0:
        intersection_point = intersection[0]
        intersection_date = future_index[intersection_point]

        print(f"The predicted intersection date is: {intersection_date.strftime('%Y-%m')}")

        # Plot the intersection
        plt.figure(figsize=(14, 7))
        plt.plot(future_index, group_pred, label='Group 1 (Bass Diffusion)', linestyle='--')
        plt.plot(future_index, item_73292_pred, label='73292 (Gompertz)', linestyle='--')
        plt.axvline(intersection_date, color='red', linestyle=':', label=f'Intersection: {intersection_date.strftime("%Y-%m")}')
        plt.title('Prediction Comparison and Intersection Point')
        plt.xlabel('Date')
        plt.ylabel('Value')
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No intersection found in the predicted range.")
else:
    print("Predictions for 'Group 1' or '73292' are not available.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import mean_absolute_error

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Rename the "Item" column values explicitly including 73358 and 73359 in "Group 1"
items_to_keep = ['73292']
filtered_data.loc[~filtered_data['Item'].isin(items_to_keep + ['73358', '73359']), 'Item'] = 'Group 1'

# List of items to fit models to
categories = ['Group 1', '73292']

# Define model functions
def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

model_predictions = {}

# Fit the specified models and predict out to 2035
for category in categories:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()
    
    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Create a time index for fitting models and extending predictions
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + (2035 - category_data.index[-1].year) * 12)

    # Fit Gompertz curve for "73292" and Bass diffusion curve for "Group 1"
    if category == '73292':
        try:
            params, _ = curve_fit(gompertz, current_time_index, category_data, maxfev=10000)
            predicted = gompertz(current_time_index, *params)
            future_predicted = gompertz(future_time_index, *params)
            model_predictions[category] = future_predicted
            print(f"Gompertz model parameters for '73292': {params}")
        except Exception as e:
            print(f"Error fitting Gompertz model for '{category}': {e}")
            continue

    elif category == 'Group 1':
        try:
            params, _ = curve_fit(bass_diffusion, current_time_index, category_data, maxfev=10000)
            predicted = bass_diffusion(current_time_index, *params)
            future_predicted = bass_diffusion(future_time_index, *params)
            model_predictions[category] = future_predicted
            print(f"Bass diffusion model parameters for 'Group 1': {params}")
        except Exception as e:
            print(f"Error fitting Bass diffusion model for '{category}': {e}")
            continue

# Visualize the predictions for each category
for category in categories:
    if category in model_predictions:
        future_index = pd.date_range(
            start=category_data.index[-1] + pd.offsets.MonthEnd(1),
            periods=len(future_time_index),
            freq='M'
        )
        plt.figure(figsize=(12, 6))
        plt.plot(future_index, model_predictions[category], label=f'{category} Prediction', linestyle='--')
        plt.title(f'Predicted Values for {category}')
        plt.xlabel('Date')
        plt.ylabel('Value')
        plt.legend()
        plt.tight_layout()
        plt.show()

# Calculate the intersection point between "Group 1" and "73292"
if 'Group 1' in model_predictions and '73292' in model_predictions:
    group_pred = model_predictions['Group 1']
    item_73292_pred = model_predictions['73292']
    future_index = pd.date_range(
        start=category_data.index[-1] + pd.offsets.MonthEnd(1),
        periods=len(future_time_index),
        freq='M'
    )
    intersection = np.where(np.diff(np.sign(group_pred - item_73292_pred)))[0]

    if intersection.size > 0:
        intersection_point = intersection[0]
        intersection_date = future_index[intersection_point]

        print(f"The predicted intersection date is: {intersection_date.strftime('%Y-%m')}")

        # Plot the intersection
        plt.figure(figsize=(14, 7))
        plt.plot(future_index, group_pred, label='Group 1 (Bass Diffusion)', linestyle='--')
        plt.plot(future_index, item_73292_pred, label='73292 (Gompertz)', linestyle='--')
        plt.axvline(intersection_date, color='red', linestyle=':', label=f'Intersection: {intersection_date.strftime("%Y-%m")}')
        plt.title('Prediction Comparison and Intersection Point')
        plt.xlabel('Date')
        plt.ylabel('Value')
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No intersection found in the predicted range.")
else:
    print("Predictions for 'Group 1' or '73292' are not available.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import mean_absolute_error

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Rename the "Item" column values explicitly including 73358 and 73359 in "Group 1"
items_to_keep = ['73292']
filtered_data.loc[~filtered_data['Item'].isin(items_to_keep + ['73358', '73359']), 'Item'] = 'Group 1'

# List of items to fit models to
categories = ['Group 1', '73292']

# Define model functions
def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

model_predictions = {}

# Fit the specified models and predict out to 2035
for category in categories:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()
    
    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Create a time index for fitting models and extending predictions
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + (2035 - category_data.index[-1].year) * 12)

    # Fit Gompertz curve for "73292" and Bass diffusion curve for "Group 1"
    if category == '73292':
        try:
            params, _ = curve_fit(gompertz, current_time_index, category_data, maxfev=10000)
            predicted = gompertz(current_time_index, *params)
            future_predicted = gompertz(future_time_index, *params)
            model_predictions[category] = future_predicted
            print(f"Gompertz model parameters for '73292': {params}")
        except Exception as e:
            print(f"Error fitting Gompertz model for '{category}': {e}")
            continue

    elif category == 'Group 1':
        try:
            params, _ = curve_fit(bass_diffusion, current_time_index, category_data, maxfev=10000)
            predicted = bass_diffusion(current_time_index, *params)
            future_predicted = bass_diffusion(future_time_index, *params)
            model_predictions[category] = future_predicted
            print(f"Bass diffusion model parameters for 'Group 1': {params}")
        except Exception as e:
            print(f"Error fitting Bass diffusion model for '{category}': {e}")
            continue

# Visualize the predictions for "Group 1" and "73292" on the same plot
if 'Group 1' in model_predictions and '73292' in model_predictions:
    group_pred = model_predictions['Group 1']
    item_73292_pred = model_predictions['73292']
    future_index = pd.date_range(
        start=category_data.index[-1] + pd.offsets.MonthEnd(1),
        periods=len(group_pred),
        freq='M'
    )

    plt.figure(figsize=(14, 7))
    plt.plot(future_index, group_pred, label='Group 1 (Bass Diffusion)', linestyle='--')
    plt.plot(future_index, item_73292_pred, label='73292 (Gompertz)', linestyle='--')
    plt.title('Prediction Comparison for Group 1 and 73292')
    plt.xlabel('Date')
    plt.ylabel('Value')
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Calculate and indicate intersection point
    intersection = np.where(np.diff(np.sign(group_pred - item_73292_pred)))[0]
    if intersection.size > 0:
        intersection_point = intersection[0]
        intersection_date = future_index[intersection_point]
        plt.axvline(intersection_date, color='red', linestyle=':', label=f'Intersection: {intersection_date.strftime("%Y-%m")}')
        print(f"The predicted intersection date is: {intersection_date.strftime('%Y-%m')}")
else:
    print("Predictions for 'Group 1' or '73292' are not available.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import mean_absolute_error

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Rename the "Item" column values explicitly including 73358 and 73359 in "Group 1"
items_to_keep = ['73292']
filtered_data.loc[~filtered_data['Item'].isin(items_to_keep + ['73358', '73359']), 'Item'] = 'Group 1'

# List of items to fit models to
categories = ['Group 1', '73292']

# Define model functions
def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

model_predictions = {}

# Fit the specified models and predict out to 2035
historical_data = {}
for category in categories:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()
    
    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Create a time index for fitting models and extending predictions
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + (2035 - category_data.index[-1].year) * 12)

    # Fit Gompertz curve for "73292" and Bass diffusion curve for "Group 1"
    if category == '73292':
        try:
            params, _ = curve_fit(gompertz, current_time_index, category_data, maxfev=10000)
            predicted = gompertz(current_time_index, *params)
            future_predicted = gompertz(future_time_index, *params)
            model_predictions[category] = future_predicted
            print(f"Gompertz model parameters for '73292': {params}")
        except Exception as e:
            print(f"Error fitting Gompertz model for '{category}': {e}")
            continue

    elif category == 'Group 1':
        try:
            params, _ = curve_fit(bass_diffusion, current_time_index, category_data, maxfev=10000)
            predicted = bass_diffusion(current_time_index, *params)
            future_predicted = bass_diffusion(future_time_index, *params)
            model_predictions[category] = future_predicted
            print(f"Bass diffusion model parameters for 'Group 1': {params}")
        except Exception as e:
            print(f"Error fitting Bass diffusion model for '{category}': {e}")
            continue

# Visualize the predictions for "Group 1" and "73292" on the same plot
if 'Group 1' in model_predictions and '73292' in model_predictions:
    group_pred = model_predictions['Group 1']
    item_73292_pred = model_predictions['73292']
    future_index = pd.date_range(
        start=historical_data['Group 1'].index[-1] + pd.offsets.MonthEnd(1),
        periods=len(group_pred),
        freq='M'
    )

    plt.figure(figsize=(14, 7))

    # Plot historical data
    plt.plot(historical_data['Group 1'].index, historical_data['Group 1'], label='Group 1 (Historical)', linestyle='-')
    plt.plot(historical_data['73292'].index, historical_data['73292'], label='73292 (Historical)', linestyle='-')

    # Plot predicted data
    plt.plot(future_index, group_pred, label='Group 1 (Predicted)', linestyle='--')
    plt.plot(future_index, item_73292_pred, label='73292 (Predicted)', linestyle='--')

    plt.title('Prediction Comparison for Group 1 and 73292')
    plt.xlabel('Date')
    plt.ylabel('Value')
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Calculate and indicate intersection point
    intersection = np.where(np.diff(np.sign(group_pred - item_73292_pred)))[0]
    if intersection.size > 0:
        intersection_point = intersection[0]
        intersection_date = future_index[intersection_point]
        plt.axvline(intersection_date, color='red', linestyle=':', label=f'Intersection: {intersection_date.strftime("%Y-%m")}')
        print(f"The predicted intersection date is: {intersection_date.strftime('%Y-%m')}")
else:
    print("Predictions for 'Group 1' or '73292' are not available.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import mean_absolute_error

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Rename the "Item" column values explicitly including 73358 and 73359 in "Group 1" and defining "Group 2"
items_to_keep = ['73292']
group_2_items = ['73298', '73358', '73359', '73401', '73402', '73422']
filtered_data.loc[~filtered_data['Item'].isin(items_to_keep + ['73358', '73359']), 'Item'] = 'Group 1'
filtered_data.loc[filtered_data['Item'].isin(group_2_items), 'Item'] = 'Group 2'

# List of items to fit models to
categories = ['Group 1', 'Group 2', '73292']

# Define model functions
def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

model_predictions = {}

# Fit the specified models and predict out to 2035
historical_data = {}
for category in categories:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()
    
    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Create a time index for fitting models and extending predictions
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + (2035 - category_data.index[-1].year) * 12)

    # Fit Gompertz curve for "73292" and Bass diffusion curve for groups
    if category == '73292':
        try:
            params, _ = curve_fit(gompertz, current_time_index, category_data, maxfev=10000)
            predicted = gompertz(current_time_index, *params)
            future_predicted = gompertz(future_time_index, *params)
            model_predictions[category] = future_predicted
            print(f"Gompertz model parameters for '73292': {params}")
        except Exception as e:
            print(f"Error fitting Gompertz model for '{category}': {e}")
            continue

    elif category in ['Group 1', 'Group 2']:
        try:
            params, _ = curve_fit(bass_diffusion, current_time_index, category_data, maxfev=10000)
            predicted = bass_diffusion(current_time_index, *params)
            future_predicted = bass_diffusion(future_time_index, *params)
            model_predictions[category] = future_predicted
            print(f"Bass diffusion model parameters for '{category}': {params}")
        except Exception as e:
            print(f"Error fitting Bass diffusion model for '{category}': {e}")
            continue

# Visualize the predictions for "Group 1", "Group 2", and "73292" on the same plot
if all(cat in model_predictions for cat in ['Group 1', 'Group 2', '73292']):
    future_index = pd.date_range(
        start=historical_data['Group 1'].index[-1] + pd.offsets.MonthEnd(1),
        periods=len(model_predictions['Group 1']),
        freq='M'
    )

    plt.figure(figsize=(14, 7))

    # Plot historical data
    for category in ['Group 1', 'Group 2', '73292']:
        plt.plot(historical_data[category].index, historical_data[category], label=f'{category} (Historical)', linestyle='-')

    # Plot predicted data
    plt.plot(future_index, model_predictions['Group 1'], label='Group 1 (Predicted)', linestyle='--')
    plt.plot(future_index, model_predictions['Group 2'], label='Group 2 (Predicted)', linestyle='--')
    plt.plot(future_index, model_predictions['73292'], label='73292 (Predicted)', linestyle='--')

#    plt.title('Prediction Comparison for Group 1, Group 2, and 73292')
    plt.xlabel('Date')
    plt.ylabel('Value')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Predictions for one or more categories are not available.")


In [ ]:
# Calculate total values for Group 1, Group 2, and 73292
total_73292 = historical_data['73292'].sum()

if total_73292 > 0:
    group_1_total = historical_data['Group 1'].sum()
    group_2_total = historical_data['Group 2'].sum()

    group_1_percentage = (group_1_total / total_73292) * 100
    group_2_percentage = (group_2_total / total_73292) * 100

    print(f"Group 1 represents {group_1_percentage:.2f}% of the value of 73292.")
    print(f"Group 2 represents {group_2_percentage:.2f}% of the value of 73292.")
else:
    print("Total value for 73292 is zero, percentages cannot be calculated.")

# Visualize the predictions for "Group 1", "Group 2", and "73292" on the same plot
if all(cat in model_predictions for cat in ['Group 1', 'Group 2', '73292']):
    future_index = pd.date_range(
        start=historical_data['Group 1'].index[-1] + pd.offsets.MonthEnd(1),
        periods=len(model_predictions['Group 1']),
        freq='M'
    )

    plt.figure(figsize=(14, 7))

    # Plot historical data
    for category in ['Group 1', 'Group 2', '73292']:
        plt.plot(historical_data[category].index, historical_data[category], label=f'{category} (Historical)', linestyle='-')

    # Plot predicted data
    plt.plot(future_index, model_predictions['Group 1'], label='Group 1 (Predicted)', linestyle='--')
    plt.plot(future_index, model_predictions['Group 2'], label='Group 2 (Predicted)', linestyle='--')
    plt.plot(future_index, model_predictions['73292'], label='73292 (Predicted)', linestyle='--')

    # Annotate the percentages on the graph
    plt.text(
        0.05, 0.95, 
        f"Group 1: {group_1_percentage:.2f}%\nGroup 2: {group_2_percentage:.2f}%",
        transform=plt.gca().transAxes,
        fontsize=12,
        verticalalignment='top',
        bbox=dict(boxstyle="round,pad=0.3", edgecolor="black", facecolor="white")
    )

    plt.xlabel('Date')
    plt.ylabel('Value')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Predictions for one or more categories are not available.")


In [ ]:
# Calculate monthly percentages for Group 1 and Group 2 relative to 73292
monthly_percentage = pd.DataFrame(index=historical_data['73292'].index)

if not historical_data['73292'].empty:
    monthly_percentage['Group 1'] = (
        historical_data['Group 1'] / historical_data['73292']
    ) * 100
    monthly_percentage['Group 2'] = (
        historical_data['Group 2'] / historical_data['73292']
    ) * 100
else:
    print("No data for '73292'. Cannot calculate percentages.")

# Ensure missing months do not cause issues
monthly_percentage = monthly_percentage.fillna(0)

# Visualize the monthly percentage trends
plt.figure(figsize=(14, 7))

# Plot percentages over time
for category in ['Group 1', 'Group 2']:
    plt.plot(
        monthly_percentage.index,
        monthly_percentage[category],
        label=f'{category} Percentage (Monthly)',
        linestyle='-'
    )

plt.xlabel('Date')
plt.ylabel('Percentage (%)')
plt.title('Monthly Percentage of Group 1 and Group 2 Relative to 73292')
plt.legend()
plt.tight_layout()
plt.show()

# Visualize the predictions for "Group 1", "Group 2", and "73292" on the same plot
if all(cat in model_predictions for cat in ['Group 1', 'Group 2', '73292']):
    future_index = pd.date_range(
        start=historical_data['Group 1'].index[-1] + pd.offsets.MonthEnd(1),
        periods=len(model_predictions['Group 1']),
        freq='M'
    )

    plt.figure(figsize=(14, 7))

    # Plot historical data
    for category in ['Group 1', 'Group 2', '73292']:
        plt.plot(historical_data[category].index, historical_data[category], label=f'{category} (Historical)', linestyle='-')

    # Plot predicted data
    plt.plot(future_index, model_predictions['Group 1'], label='Group 1 (Predicted)', linestyle='--')
    plt.plot(future_index, model_predictions['Group 2'], label='Group 2 (Predicted)', linestyle='--')
    plt.plot(future_index, model_predictions['73292'], label='73292 (Predicted)', linestyle='--')

    plt.xlabel('Date')
    plt.ylabel('Value')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Predictions for one or more categories are not available.")


In [ ]:
# Calculate monthly percentages for historical data
monthly_percentage = pd.DataFrame(index=historical_data['73292'].index)

if not historical_data['73292'].empty:
    monthly_percentage['Group 1'] = (
        historical_data['Group 1'] / historical_data['73292']
    ) * 100
    monthly_percentage['Group 2'] = (
        historical_data['Group 2'] / historical_data['73292']
    ) * 100
else:
    print("No data for '73292'. Cannot calculate historical percentages.")

# Ensure missing months do not cause issues
monthly_percentage = monthly_percentage.fillna(0)

# Extend percentages into predictions
if all(cat in model_predictions for cat in ['Group 1', 'Group 2', '73292']):
    # Create a DataFrame for predicted percentages
    future_index = pd.date_range(
        start=historical_data['Group 1'].index[-1] + pd.offsets.MonthEnd(1),
        periods=len(model_predictions['Group 1']),
        freq='M'
    )
    predicted_percentage = pd.DataFrame(index=future_index)

    # Calculate predicted percentages
    predicted_percentage['Group 1'] = (
        model_predictions['Group 1'] / model_predictions['73292']
    ) * 100
    predicted_percentage['Group 2'] = (
        model_predictions['Group 2'] / model_predictions['73292']
    ) * 100
else:
    print("Predictions for one or more categories are not available.")
    predicted_percentage = None

# Combine historical and predicted percentages for visualization
if predicted_percentage is not None:
    combined_percentage = pd.concat([monthly_percentage, predicted_percentage])
else:
    combined_percentage = monthly_percentage

# Visualize the combined percentage trends
plt.figure(figsize=(14, 7))

for category in ['Group 1', 'Group 2']:
    plt.plot(
        combined_percentage.index,
        combined_percentage[category],
        label=f'{category} Percentage',
        linestyle='-'
    )

plt.xlabel('Date')
plt.ylabel('Percentage (%)')
plt.title('Percentage of Group 1 and Group 2 Relative to 73292 (Historical and Predicted)')
plt.legend()
plt.tight_layout()
plt.show()

# Visualize the predictions for "Group 1", "Group 2", and "73292" on the same plot
if all(cat in model_predictions for cat in ['Group 1', 'Group 2', '73292']):
    plt.figure(figsize=(14, 7))

    # Plot historical data
    for category in ['Group 1', 'Group 2', '73292']:
        plt.plot(historical_data[category].index, historical_data[category], label=f'{category} (Historical)', linestyle='-')

    # Plot predicted data
    future_index = pd.date_range(
        start=historical_data['Group 1'].index[-1] + pd.offsets.MonthEnd(1),
        periods=len(model_predictions['Group 1']),
        freq='M'
    )
    plt.plot(future_index, model_predictions['Group 1'], label='Group 1 (Predicted)', linestyle='--')
    plt.plot(future_index, model_predictions['Group 2'], label='Group 2 (Predicted)', linestyle='--')
    plt.plot(future_index, model_predictions['73292'], label='73292 (Predicted)', linestyle='--')

    plt.xlabel('Date')
    plt.ylabel('Value')
    plt.title('Historical and Predicted Values for Group 1, Group 2, and 73292')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Predictions for one or more categories are not available.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import mean_absolute_error

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Rename the "Item" column values explicitly including 73358 and 73359 in "Group 1" and defining "Group 2"
items_to_keep = ['73292']
group_2_items = ['73298', '73358', '73359', '73401', '73402', '73422']
filtered_data.loc[~filtered_data['Item'].isin(items_to_keep + ['73358', '73359']), 'Item'] = 'Group 1'
filtered_data.loc[filtered_data['Item'].isin(group_2_items), 'Item'] = 'Group 2'

# List of items to fit models to
categories = ['Group 1', 'Group 2', '73292']

# Define model functions
def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

model_predictions = {}

# Fit the specified models and predict out to 2035
historical_data = {}
for category in categories:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()
    
    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Create a time index for fitting models and extending predictions
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + (2035 - category_data.index[-1].year) * 12)

    # Fit Gompertz curve for "73292" and Bass diffusion curve for groups
    if category == '73292':
        try:
            params, _ = curve_fit(gompertz, current_time_index, category_data, maxfev=10000)
            future_predicted = gompertz(future_time_index, *params)
            model_predictions[category] = future_predicted
            print(f"Gompertz model parameters for '73292': {params}")
        except Exception as e:
            print(f"Error fitting Gompertz model for '{category}': {e}")
            continue

    elif category in ['Group 1', 'Group 2']:
        try:
            params, _ = curve_fit(bass_diffusion, current_time_index, category_data, maxfev=10000)
            future_predicted = bass_diffusion(future_time_index, *params)
            model_predictions[category] = future_predicted
            print(f"Bass diffusion model parameters for '{category}': {params}")
        except Exception as e:
            print(f"Error fitting Bass diffusion model for '{category}': {e}")
            continue

# Calculate monthly percentages for historical data
monthly_percentage = pd.DataFrame(index=historical_data['73292'].index)

if not historical_data['73292'].empty:
    monthly_percentage['Group 1'] = (
        historical_data['Group 1'] / historical_data['73292']
    ) * 100
    monthly_percentage['Group 2'] = (
        historical_data['Group 2'] / historical_data['73292']
    ) * 100
else:
    print("No data for '73292'. Cannot calculate historical percentages.")

# Ensure missing months do not cause issues
monthly_percentage = monthly_percentage.fillna(0)

# Extend percentages into predictions
if all(cat in model_predictions for cat in ['Group 1', 'Group 2', '73292']):
    # Create a DataFrame for predicted percentages
    future_index = pd.date_range(
        start=historical_data['Group 1'].index[-1] + pd.offsets.MonthEnd(1),
        periods=len(model_predictions['Group 1']),
        freq='M'
    )
    predicted_percentage = pd.DataFrame(index=future_index)

    # Calculate predicted percentages
    predicted_percentage['Group 1'] = (
        model_predictions['Group 1'] / model_predictions['73292']
    ) * 100
    predicted_percentage['Group 2'] = (
        model_predictions['Group 2'] / model_predictions['73292']
    ) * 100
else:
    print("Predictions for one or more categories are not available.")
    predicted_percentage = None

# Combine historical and predicted percentages for visualization
if predicted_percentage is not None:
    combined_percentage = pd.concat([monthly_percentage, predicted_percentage])
else:
    combined_percentage = monthly_percentage

# Visualize the combined percentage trends
plt.figure(figsize=(14, 7))

for category in ['Group 1', 'Group 2']:
    plt.plot(
        combined_percentage.index,
        combined_percentage[category],
        label=f'{category} Percentage',
        linestyle='-'
    )

plt.xlabel('Date')
plt.ylabel('Percentage (%)')
plt.title('Percentage of Group 1 and Group 2 Relative to 73292 (Historical and Predicted)')
plt.legend()
plt.tight_layout()
plt.show()

# Visualize the predictions for "Group 1", "Group 2", and "73292" on the same plot
if all(cat in model_predictions for cat in ['Group 1', 'Group 2', '73292']):
    plt.figure(figsize=(14, 7))

    # Plot historical data
    for category in ['Group 1', 'Group 2', '73292']:
        plt.plot(historical_data[category].index, historical_data[category], label=f'{category} (Historical)', linestyle='-')

    # Plot predicted data
    future_index = pd.date_range(
        start=historical_data['Group 1'].index[-1] + pd.offsets.MonthEnd(1),
        periods=len(model_predictions['Group 1']),
        freq='M'
    )
    plt.plot(future_index, model_predictions['Group 1'], label='Group 1 (Predicted)', linestyle='--')
    plt.plot(future_index, model_predictions['Group 2'], label='Group 2 (Predicted)', linestyle='--')
    plt.plot(future_index, model_predictions['73292'], label='73292 (Predicted)', linestyle='--')

    plt.xlabel('Date')
    plt.ylabel('Value')
    plt.title('Historical and Predicted Values for Group 1, Group 2, and 73292')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Predictions for one or more categories are not available.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Define the individual components of each group
group_1_items = [item for item in filtered_data['Item'].unique() if item not in ['73292', '73358', '73359']]
group_2_items = ['73358', '73359']

# Define model functions
def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

# Fit models for each individual component and predict future utilization
model_predictions = {}
historical_data = {}
aggregated_predictions = {'Group 1': None, 'Group 2': None}

for category in ['73292'] + group_1_items + group_2_items:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()
    
    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Create a time index for fitting models and extending predictions
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + (2035 - category_data.index[-1].year) * 12)

    # Fit appropriate model (Gompertz for 73292, Bass diffusion for groups)
    try:
        if category == '73292':
            params, _ = curve_fit(gompertz, current_time_index, category_data, maxfev=10000)
            future_predicted = gompertz(future_time_index, *params)
        else:
            params, _ = curve_fit(bass_diffusion, current_time_index, category_data, maxfev=10000)
            future_predicted = bass_diffusion(future_time_index, *params)

        model_predictions[category] = future_predicted
        print(f"Model parameters for '{category}': {params}")
    except Exception as e:
        print(f"Error fitting model for '{category}': {e}")
        continue

# Aggregate predictions for Group 1 and Group 2
if all(item in model_predictions for item in group_1_items):
    aggregated_predictions['Group 1'] = np.sum([model_predictions[item] for item in group_1_items], axis=0)

if all(item in model_predictions for item in group_2_items):
    aggregated_predictions['Group 2'] = np.sum([model_predictions[item] for item in group_2_items], axis=0)

# Calculate percentages relative to 73292 for historical and predicted data
if '73292' in model_predictions:
    future_index = pd.date_range(
        start=historical_data['73292'].index[-1] + pd.offsets.MonthEnd(1),
        periods=len(model_predictions['73292']),
        freq='M'
    )
    predicted_percentage = pd.DataFrame(index=future_index)
    predicted_percentage['Group 1'] = (
        aggregated_predictions['Group 1'] / model_predictions['73292']
    ) * 100
    predicted_percentage['Group 2'] = (
        aggregated_predictions['Group 2'] / model_predictions['73292']
    ) * 100
else:
    predicted_percentage = None

# Visualize predictions and percentages
if predicted_percentage is not None:
    plt.figure(figsize=(14, 7))
    plt.plot(future_index, predicted_percentage['Group 1'], label='Group 1 Percentage (Predicted)', linestyle='--')
    plt.plot(future_index, predicted_percentage['Group 2'], label='Group 2 Percentage (Predicted)', linestyle='--')

    plt.xlabel('Date')
    plt.ylabel('Percentage (%)')
    plt.title('Predicted Percentage of Group 1 and Group 2 Relative to 73292')
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(14, 7))
    plt.plot(future_index, aggregated_predictions['Group 1'], label='Group 1 (Aggregated)', linestyle='--')
    plt.plot(future_index, aggregated_predictions['Group 2'], label='Group 2 (Aggregated)', linestyle='--')
    plt.plot(future_index, model_predictions['73292'], label='73292 (Predicted)', linestyle='--')

    plt.xlabel('Date')
    plt.ylabel('Value')
    plt.title('Predicted Values for Group 1, Group 2, and 73292')
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Define the individual components of each group
group_1_items = [item for item in filtered_data['Item'].unique() if item not in ['73292', '73358', '73359']]
group_2_items = ['73358', '73359']

# Define model functions
def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

# Fit models for each individual component and predict future utilization
model_predictions = {}
historical_data = {}
aggregated_predictions = {'Group 1': None, 'Group 2': None}

for category in ['73292'] + group_1_items + group_2_items:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()
    
    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Create a time index for fitting models and extending predictions
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + (2035 - category_data.index[-1].year) * 12)

    # Fit appropriate model (Gompertz for 73292, Bass diffusion for groups)
    try:
        if category == '73292':
            params, _ = curve_fit(gompertz, current_time_index, category_data, maxfev=10000)
            future_predicted = gompertz(future_time_index, *params)
        else:
            params, _ = curve_fit(bass_diffusion, current_time_index, category_data, maxfev=10000)
            future_predicted = bass_diffusion(future_time_index, *params)

        model_predictions[category] = future_predicted
        print(f"Model parameters for '{category}': {params}")
    except Exception as e:
        print(f"Error fitting model for '{category}': {e}")
        continue

# Aggregate predictions for Group 1 and Group 2
if all(item in model_predictions for item in group_1_items):
    aggregated_predictions['Group 1'] = np.sum([model_predictions[item] for item in group_1_items], axis=0)

if all(item in model_predictions for item in group_2_items):
    aggregated_predictions['Group 2'] = np.sum([model_predictions[item] for item in group_2_items], axis=0)

# Create future index for plotting predictions
if '73292' in model_predictions:
    future_index = pd.date_range(
        start=historical_data['73292'].index[-1] + pd.offsets.MonthEnd(1),
        periods=len(model_predictions['73292']),
        freq='M'
    )

# Plotting original and predicted values
plt.figure(figsize=(14, 7))

# Plot historical data for 73292, Group 1, and Group 2
plt.plot(historical_data['73292'].index, historical_data['73292'], label='73292 (Historical)', linestyle='-')
if aggregated_predictions['Group 1'] is not None:
    group_1_historical = historical_data[group_1_items[0]].copy()
    for item in group_1_items[1:]:
        group_1_historical += historical_data[item]
    plt.plot(group_1_historical.index, group_1_historical, label='Group 1 (Historical)', linestyle='-')

if aggregated_predictions['Group 2'] is not None:
    group_2_historical = historical_data[group_2_items[0]].copy()
    for item in group_2_items[1:]:
        group_2_historical += historical_data[item]
    plt.plot(group_2_historical.index, group_2_historical, label='Group 2 (Historical)', linestyle='-')

# Plot predicted data for 73292, Group 1, and Group 2
if '73292' in model_predictions:
    plt.plot(future_index, model_predictions['73292'], label='73292 (Predicted)', linestyle='--')

if aggregated_predictions['Group 1'] is not None:
    plt.plot(future_index, aggregated_predictions['Group 1'], label='Group 1 (Predicted)', linestyle='--')

if aggregated_predictions['Group 2'] is not None:
    plt.plot(future_index, aggregated_predictions['Group 2'], label='Group 2 (Predicted)', linestyle='--')

# Finalize the plot
plt.xlabel('Date')
plt.ylabel('Value')
plt.title('Historical and Predicted Values for 73292, Group 1, and Group 2')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.utils import resample

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Define the individual components of each group
group_1_items = [item for item in filtered_data['Item'].unique() if item not in ['73292', '73358', '73359']]
group_2_items = ['73358', '73359']

# Define model functions
def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

# Fit models for each individual component and predict future utilization
model_predictions = {}
historical_data = {}
aggregated_predictions = {'Group 1': None, 'Group 2': None}
bootstrap_ci = {}

for category in ['73292'] + group_1_items + group_2_items:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()
    
    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Create a time index for fitting models and extending predictions
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + (2035 - category_data.index[-1].year) * 12)

    # Fit appropriate model (Gompertz for 73292, Bass diffusion for groups)
    try:
        if category == '73292':
            params, _ = curve_fit(gompertz, current_time_index, category_data, maxfev=10000)
            future_predicted = gompertz(future_time_index, *params)
        else:
            params, _ = curve_fit(bass_diffusion, current_time_index, category_data, maxfev=10000)
            future_predicted = bass_diffusion(future_time_index, *params)

        model_predictions[category] = future_predicted

        # Bootstrapping for confidence intervals
        bootstrapped_fits = []
        for _ in range(1000):  # 1000 bootstrap iterations
            resampled_data = resample(category_data.values)
            try:
                if category == '73292':
                    boot_params, _ = curve_fit(gompertz, current_time_index, resampled_data, maxfev=10000)
                    bootstrapped_fits.append(gompertz(future_time_index, *boot_params))
                else:
                    boot_params, _ = curve_fit(bass_diffusion, current_time_index, resampled_data, maxfev=10000)
                    bootstrapped_fits.append(bass_diffusion(future_time_index, *boot_params))
            except Exception:
                continue

        bootstrapped_fits = np.array(bootstrapped_fits)
        ci_lower = np.percentile(bootstrapped_fits, 2.5, axis=0)
        ci_upper = np.percentile(bootstrapped_fits, 97.5, axis=0)
        bootstrap_ci[category] = (ci_lower, ci_upper)

        print(f"Model parameters for '{category}': {params}")
    except Exception as e:
        print(f"Error fitting model for '{category}': {e}")
        continue

# Aggregate predictions for Group 1 and Group 2
if all(item in model_predictions for item in group_1_items):
    aggregated_predictions['Group 1'] = np.sum([model_predictions[item] for item in group_1_items], axis=0)

if all(item in model_predictions for item in group_2_items):
    aggregated_predictions['Group 2'] = np.sum([model_predictions[item] for item in group_2_items], axis=0)

# Create future index for plotting predictions
if '73292' in model_predictions:
    future_index = pd.date_range(
        start=historical_data['73292'].index[-1] + pd.offsets.MonthEnd(1),
        periods=len(model_predictions['73292']),
        freq='M'
    )

# Plotting historical and predicted values with confidence intervals
plt.figure(figsize=(14, 7))

for category, label in zip(['73292', 'Group 1', 'Group 2'], ['73292', 'Group 1', 'Group 2']):
    if category == 'Group 1':
        ci_lower = np.sum([bootstrap_ci[item][0] for item in group_1_items], axis=0)
        ci_upper = np.sum([bootstrap_ci[item][1] for item in group_1_items], axis=0)
    elif category == 'Group 2':
        ci_lower = np.sum([bootstrap_ci[item][0] for item in group_2_items], axis=0)
        ci_upper = np.sum([bootstrap_ci[item][1] for item in group_2_items], axis=0)
    else:
        ci_lower, ci_upper = bootstrap_ci[category]

    # Plot historical data
    if category in historical_data:
        plt.plot(historical_data[category].index, historical_data[category], label=f'{label} (Historical)', linestyle='-')

    # Plot predicted data with confidence intervals
    if category in model_predictions:
        plt.plot(future_index, model_predictions[category], label=f'{label} (Predicted)', linestyle='--')
        plt.fill_between(future_index, ci_lower, ci_upper, alpha=0.2, label=f'{label} (95% CI)')

# Finalize the plot
plt.xlabel('Date')
plt.ylabel('Value')
plt.title('Historical and Predicted Values with 95% Confidence Intervals')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.utils import resample

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Define the individual components of each group
group_1_items = [item for item in filtered_data['Item'].unique() if item not in ['73292', '73358', '73359']]
group_2_items = ['73358', '73359']

# Define model functions
def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

# Fit models for each individual component and generate predictions
model_predictions = {}
historical_data = {}
aggregated_predictions = {'Group 1': None, 'Group 2': None}
bootstrap_ci = {}

for category in ['73292'] + group_1_items + group_2_items:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Create a time index for fitting models and extending predictions
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + (2035 - category_data.index[-1].year) * 12)

    try:
        # Fit appropriate model
        if category == '73292':
            params, _ = curve_fit(gompertz, current_time_index, category_data, maxfev=10000)
            fitted_values = gompertz(current_time_index, *params)
            future_predicted = gompertz(future_time_index, *params)
        else:
            params, _ = curve_fit(bass_diffusion, current_time_index, category_data, maxfev=10000)
            fitted_values = bass_diffusion(current_time_index, *params)
            future_predicted = bass_diffusion(future_time_index, *params)

        model_predictions[category] = future_predicted

        # Calculate residuals
        residuals = category_data.values - fitted_values

        # Bootstrapping for confidence intervals
        bootstrapped_predictions = []
        for _ in range(1000):  # 1000 bootstrap iterations
            resampled_residuals = resample(residuals)
            simulated_data = fitted_values + resampled_residuals
            try:
                if category == '73292':
                    boot_params, _ = curve_fit(gompertz, current_time_index, simulated_data, maxfev=10000)
                    bootstrapped_predictions.append(gompertz(future_time_index, *boot_params))
                else:
                    boot_params, _ = curve_fit(bass_diffusion, current_time_index, simulated_data, maxfev=10000)
                    bootstrapped_predictions.append(bass_diffusion(future_time_index, *boot_params))
            except Exception:
                continue

        bootstrapped_predictions = np.array(bootstrapped_predictions)
        ci_lower = np.percentile(bootstrapped_predictions, 2.5, axis=0)
        ci_upper = np.percentile(bootstrapped_predictions, 97.5, axis=0)
        bootstrap_ci[category] = (ci_lower, ci_upper)

        print(f"Model parameters for '{category}': {params}")
    except Exception as e:
        print(f"Error fitting model for '{category}': {e}")
        continue

# Aggregate predictions for Group 1 and Group 2
if all(item in model_predictions for item in group_1_items):
    aggregated_predictions['Group 1'] = np.sum([model_predictions[item] for item in group_1_items], axis=0)

if all(item in model_predictions for item in group_2_items):
    aggregated_predictions['Group 2'] = np.sum([model_predictions[item] for item in group_2_items], axis=0)

# Create future index for plotting predictions
if '73292' in model_predictions:
    future_index = pd.date_range(
        start=historical_data['73292'].index[-1] + pd.offsets.MonthEnd(1),
        periods=len(model_predictions['73292']),
        freq='M'
    )

# Plotting historical and predicted values with confidence intervals
plt.figure(figsize=(14, 7))

for category, label in zip(['73292', 'Group 1', 'Group 2'], ['73292', 'Group 1', 'Group 2']):
    if category == 'Group 1':
        ci_lower = np.sum([bootstrap_ci[item][0] for item in group_1_items], axis=0)
        ci_upper = np.sum([bootstrap_ci[item][1] for item in group_1_items], axis=0)
    elif category == 'Group 2':
        ci_lower = np.sum([bootstrap_ci[item][0] for item in group_2_items], axis=0)
        ci_upper = np.sum([bootstrap_ci[item][1] for item in group_2_items], axis=0)
    else:
        ci_lower, ci_upper = bootstrap_ci[category]

    # Plot historical data
    if category in historical_data:
        plt.plot(historical_data[category].index, historical_data[category], label=f'{label} (Historical)', linestyle='-')

    # Plot predicted data with confidence intervals
    if category in model_predictions:
        plt.plot(future_index, model_predictions[category], label=f'{label} (Predicted)', linestyle='--')
        plt.fill_between(future_index, ci_lower, ci_upper, alpha=0.2, label=f'{label} (95% CI)')

# Finalize the plot
plt.xlabel('Date')
plt.ylabel('Value')
plt.title('Historical and Predicted Values with Residual-Based 95% Confidence Intervals')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.utils import resample

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Define the individual components of each group
group_1_items = [item for item in filtered_data['Item'].unique() if item not in ['73292', '73358', '73359']]
group_2_items = ['73358', '73359']

# Define model functions
def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

# Fit models for each individual component and generate predictions
model_predictions = {}
historical_data = {}
aggregated_predictions = {'Group 1': None, 'Group 2': None}
bootstrap_ci = {}

for category in ['73292'] + group_1_items + group_2_items:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Create a time index for fitting models and extending predictions
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + (2035 - category_data.index[-1].year) * 12)

    try:
        # Fit appropriate model
        if category == '73292':
            params, _ = curve_fit(gompertz, current_time_index, category_data, maxfev=10000)
            fitted_values = gompertz(current_time_index, *params)
            future_predicted = gompertz(future_time_index, *params)
        else:
            params, _ = curve_fit(bass_diffusion, current_time_index, category_data, maxfev=10000)
            fitted_values = bass_diffusion(current_time_index, *params)
            future_predicted = bass_diffusion(future_time_index, *params)

        model_predictions[category] = future_predicted

        # Calculate residuals
        residuals = category_data.values - fitted_values

        # Bootstrapping for confidence intervals
        bootstrapped_predictions = []
        for _ in range(1000):  # 1000 bootstrap iterations
            resampled_residuals = resample(residuals)
            simulated_data = fitted_values + resampled_residuals
            try:
                if category == '73292':
                    boot_params, _ = curve_fit(gompertz, current_time_index, simulated_data, maxfev=10000)
                    bootstrapped_predictions.append(gompertz(future_time_index, *boot_params))
                else:
                    boot_params, _ = curve_fit(bass_diffusion, current_time_index, simulated_data, maxfev=10000)
                    bootstrapped_predictions.append(bass_diffusion(future_time_index, *boot_params))
            except Exception:
                continue

        bootstrapped_predictions = np.array(bootstrapped_predictions)
        ci_lower = np.percentile(bootstrapped_predictions, 2.5, axis=0)
        ci_upper = np.percentile(bootstrapped_predictions, 97.5, axis=0)
        bootstrap_ci[category] = (ci_lower, ci_upper)

        print(f"Model parameters for '{category}': {params}")
    except Exception as e:
        print(f"Error fitting model for '{category}': {e}")
        continue

# Aggregate predictions and confidence intervals for Group 1 and Group 2
if all(item in model_predictions for item in group_1_items):
    aggregated_predictions['Group 1'] = np.sum([model_predictions[item] for item in group_1_items], axis=0)
    ci_lower_group_1 = np.sum([bootstrap_ci[item][0] for item in group_1_items], axis=0)
    ci_upper_group_1 = np.sum([bootstrap_ci[item][1] for item in group_1_items], axis=0)

if all(item in model_predictions for item in group_2_items):
    aggregated_predictions['Group 2'] = np.sum([model_predictions[item] for item in group_2_items], axis=0)
    ci_lower_group_2 = np.sum([bootstrap_ci[item][0] for item in group_2_items], axis=0)
    ci_upper_group_2 = np.sum([bootstrap_ci[item][1] for item in group_2_items], axis=0)

# Create future index for plotting predictions
if '73292' in model_predictions:
    future_index = pd.date_range(
        start=historical_data['73292'].index[-1] + pd.offsets.MonthEnd(1),
        periods=len(model_predictions['73292']),
        freq='M'
    )

# Plotting historical and predicted values with confidence intervals
plt.figure(figsize=(14, 7))

# Plot 73292
if '73292' in model_predictions:
    ci_lower, ci_upper = bootstrap_ci['73292']
    plt.plot(historical_data['73292'].index, historical_data['73292'], label='73292 (Historical)', linestyle='-')
    plt.plot(future_index, model_predictions['73292'], label='73292 (Predicted)', linestyle='--')
    plt.fill_between(future_index, ci_lower, ci_upper, alpha=0.2, label='73292 (95% CI)')

# Plot Group 1
if aggregated_predictions['Group 1'] is not None:
    plt.plot(future_index, aggregated_predictions['Group 1'], label='Group 1 (Predicted)', linestyle='--')
    plt.fill_between(future_index, ci_lower_group_1, ci_upper_group_1, alpha=0.2, label='Group 1 (95% CI)')

# Plot Group 2
if aggregated_predictions['Group 2'] is not None:
    plt.plot(future_index, aggregated_predictions['Group 2'], label='Group 2 (Predicted)', linestyle='--')
    plt.fill_between(future_index, ci_lower_group_2, ci_upper_group_2, alpha=0.2, label='Group 2 (95% CI)')

# Finalize the plot
plt.xlabel('Date')
plt.ylabel('Value')
plt.title('Historical and Predicted Values with Bootstrapped 95% Confidence Intervals')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.utils import resample

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Define the individual components of each group
group_1_items = [item for item in filtered_data['Item'].unique() if item not in ['73292', '73358', '73359']]
group_2_items = ['73358', '73359']

# Define model functions
def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

# Fit models for each individual component and generate predictions
model_predictions = {}
historical_data = {}
aggregated_predictions = {'Group 1': None, 'Group 2': None}
bootstrap_ci = {}

for category in ['73292'] + group_1_items + group_2_items:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Create a time index for fitting models and extending predictions
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + (2035 - category_data.index[-1].year) * 12)

    try:
        # Fit appropriate model
        if category == '73292':
            params, _ = curve_fit(gompertz, current_time_index, category_data, maxfev=10000)
            fitted_values = gompertz(current_time_index, *params)
            future_predicted = gompertz(future_time_index, *params)
        else:
            params, _ = curve_fit(bass_diffusion, current_time_index, category_data, maxfev=10000)
            fitted_values = bass_diffusion(current_time_index, *params)
            future_predicted = bass_diffusion(future_time_index, *params)

        model_predictions[category] = future_predicted

        # Calculate residuals
        residuals = category_data.values - fitted_values

        # Bootstrapping for confidence intervals
        bootstrapped_predictions = []
        for _ in range(1000):  # 1000 bootstrap iterations
            resampled_residuals = resample(residuals)
            simulated_data = fitted_values + resampled_residuals
            try:
                if category == '73292':
                    boot_params, _ = curve_fit(gompertz, current_time_index, simulated_data, maxfev=10000)
                    bootstrapped_predictions.append(gompertz(future_time_index, *boot_params))
                else:
                    boot_params, _ = curve_fit(bass_diffusion, current_time_index, simulated_data, maxfev=10000)
                    bootstrapped_predictions.append(bass_diffusion(future_time_index, *boot_params))
            except Exception:
                continue

        bootstrapped_predictions = np.array(bootstrapped_predictions)
        ci_lower = np.percentile(bootstrapped_predictions, 2.5, axis=0)
        ci_upper = np.percentile(bootstrapped_predictions, 97.5, axis=0)
        bootstrap_ci[category] = (ci_lower, ci_upper)

        print(f"Model parameters for '{category}': {params}")
    except Exception as e:
        print(f"Error fitting model for '{category}': {e}")
        continue

# Aggregate predictions and confidence intervals for Group 1 and Group 2
if all(item in model_predictions for item in group_1_items):
    aggregated_predictions['Group 1'] = np.sum([model_predictions[item] for item in group_1_items], axis=0)
    bootstrapped_group_1 = np.array(
        [np.sum([bootstrap_ci[item][i] for item in group_1_items], axis=0) for i in [0, 1]]
    )
    ci_lower_group_1 = bootstrapped_group_1[0]
    ci_upper_group_1 = bootstrapped_group_1[1]

if all(item in model_predictions for item in group_2_items):
    aggregated_predictions['Group 2'] = np.sum([model_predictions[item] for item in group_2_items], axis=0)
    bootstrapped_group_2 = np.array(
        [np.sum([bootstrap_ci[item][i] for item in group_2_items], axis=0) for i in [0, 1]]
    )
    ci_lower_group_2 = bootstrapped_group_2[0]
    ci_upper_group_2 = bootstrapped_group_2[1]

# Create future index for plotting predictions
if '73292' in model_predictions:
    future_index = pd.date_range(
        start=historical_data['73292'].index[-1] + pd.offsets.MonthEnd(1),
        periods=len(model_predictions['73292']),
        freq='M'
    )

# Plotting historical and predicted values with confidence intervals
plt.figure(figsize=(14, 7))

# Plot 73292
if '73292' in model_predictions:
    ci_lower, ci_upper = bootstrap_ci['73292']
    plt.plot(historical_data['73292'].index, historical_data['73292'], label='73292 (Historical)', linestyle='-')
    plt.plot(future_index, model_predictions['73292'], label='73292 (Predicted)', linestyle='--')
    plt.fill_between(future_index, ci_lower, ci_upper, alpha=0.2, label='73292 (95% CI)')

# Plot Group 1
if aggregated_predictions['Group 1'] is not None:
    plt.plot(future_index, aggregated_predictions['Group 1'], label='Group 1 (Predicted)', linestyle='--')
    plt.fill_between(future_index, ci_lower_group_1, ci_upper_group_1, alpha=0.2, label='Group 1 (95% CI)')

# Plot Group 2
if aggregated_predictions['Group 2'] is not None:
    plt.plot(future_index, aggregated_predictions['Group 2'], label='Group 2 (Predicted)', linestyle='--')
    plt.fill_between(future_index, ci_lower_group_2, ci_upper_group_2, alpha=0.2, label='Group 2 (95% CI)')

# Finalize the plot
plt.xlabel('Date')
plt.ylabel('Value')
plt.title('Historical and Predicted Values with Bootstrapped 95% Confidence Intervals')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.utils import resample

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Define the individual components of each group
group_1_items = [item for item in filtered_data['Item'].unique() if item not in ['73292', '73358', '73359']]
group_2_items = ['73358', '73359']

# Define model functions
def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

# Fit models for each individual component and generate predictions
model_predictions = {}
historical_data = {}
aggregated_predictions = {'Group 1': None, 'Group 2': None}
bootstrap_ci = {}

for category in ['73292'] + group_1_items + group_2_items:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Create a time index for fitting models and extending predictions
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + (2035 - category_data.index[-1].year) * 12)

    try:
        # Fit appropriate model
        if category == '73292':
            params, _ = curve_fit(gompertz, current_time_index, category_data, maxfev=10000)
            fitted_values = gompertz(current_time_index, *params)
            future_predicted = gompertz(future_time_index, *params)
        else:
            params, _ = curve_fit(bass_diffusion, current_time_index, category_data, maxfev=10000)
            fitted_values = bass_diffusion(current_time_index, *params)
            future_predicted = bass_diffusion(future_time_index, *params)

        model_predictions[category] = future_predicted

        # Calculate residuals
        residuals = category_data.values - fitted_values

        # Bootstrapping for confidence intervals
        bootstrapped_predictions = []
        for _ in range(1000):  # 1000 bootstrap iterations
            resampled_residuals = resample(residuals)
            simulated_data = fitted_values + resampled_residuals
            try:
                if category == '73292':
                    boot_params, _ = curve_fit(gompertz, current_time_index, simulated_data, maxfev=10000)
                    bootstrapped_predictions.append(gompertz(future_time_index, *boot_params))
                else:
                    boot_params, _ = curve_fit(bass_diffusion, current_time_index, simulated_data, maxfev=10000)
                    bootstrapped_predictions.append(bass_diffusion(future_time_index, *boot_params))
            except Exception:
                continue

        bootstrapped_predictions = np.array(bootstrapped_predictions)
        ci_lower = np.percentile(bootstrapped_predictions, 2.5, axis=0)
        ci_upper = np.percentile(bootstrapped_predictions, 97.5, axis=0)
        bootstrap_ci[category] = (ci_lower, ci_upper)

        print(f"Model parameters for '{category}': {params}")
    except Exception as e:
        print(f"Error fitting model for '{category}': {e}")
        continue

# Aggregate predictions and confidence intervals for Group 1 and Group 2
if all(item in model_predictions for item in group_1_items):
    aggregated_predictions['Group 1'] = np.sum([model_predictions[item] for item in group_1_items], axis=0)
    bootstrapped_group_1 = np.array(
        [np.sum([bootstrap_ci[item][i] for item in group_1_items], axis=0) for i in [0, 1]]
    )
    ci_lower_group_1 = bootstrapped_group_1[0]
    ci_upper_group_1 = bootstrapped_group_1[1]

if all(item in model_predictions for item in group_2_items):
    aggregated_predictions['Group 2'] = np.sum([model_predictions[item] for item in group_2_items], axis=0)
    bootstrapped_group_2 = np.array(
        [np.sum([bootstrap_ci[item][i] for item in group_2_items], axis=0) for i in [0, 1]]
    )
    ci_lower_group_2 = bootstrapped_group_2[0]
    ci_upper_group_2 = bootstrapped_group_2[1]

# Create future index for plotting predictions
if '73292' in model_predictions:
    future_index = pd.date_range(
        start=historical_data['73292'].index[-1] + pd.offsets.MonthEnd(1),
        periods=len(model_predictions['73292']),
        freq='M'
    )

# Plotting historical and predicted values with confidence intervals
plt.figure(figsize=(14, 7))

# Plot 73292
if '73292' in model_predictions:
    ci_lower, ci_upper = bootstrap_ci['73292']
    plt.plot(historical_data['73292'].index, historical_data['73292'], label='73292 (Historical)', linestyle='-')
    plt.plot(future_index, model_predictions['73292'], label='73292 (Predicted)', linestyle='--')
    plt.fill_between(future_index, ci_lower, ci_upper, alpha=0.2, label='73292 (95% CI)')

# Plot Group 1
if aggregated_predictions['Group 1'] is not None:
    plt.plot(future_index, aggregated_predictions['Group 1'], label='Group 1 (Predicted)', linestyle='--')
    plt.fill_between(future_index, ci_lower_group_1, ci_upper_group_1, alpha=0.2, label='Group 1 (95% CI)')

# Plot Group 2
if aggregated_predictions['Group 2'] is not None:
    plt.plot(future_index, aggregated_predictions['Group 2'], label='Group 2 (Predicted)', linestyle='--')
    plt.fill_between(future_index, ci_lower_group_2, ci_upper_group_2, alpha=0.2, label='Group 2 (95% CI)')

# Finalize the plot
plt.xlabel('Date')
plt.ylabel('Value')
plt.title('Historical and Predicted Values with Bootstrapped 95% Confidence Intervals')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from bootstrapped import bootstrap as bs
from bootstrapped import stats_functions as bs_stats

# Load the data
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Define the individual components of each group
group_1_items = [item for item in filtered_data['Item'].unique() if item not in ['73292', '73358', '73359']]
group_2_items = ['73358', '73359']

# Define model functions
def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

# Initialize containers
model_predictions = {}
confidence_intervals = {}
historical_data = {}

# Function to calculate bootstrapped CIs
def calculate_bootstrap_ci(data):
    ci = bs.bootstrap(data, stat_func=bs_stats.mean, alpha=0.05)
    return ci.lower_bound, ci.upper_bound

# Process each category
for category in ['73292'] + group_1_items + group_2_items:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Convert to NumPy array
    category_data = category_data.values

    # Create time indices
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + 120)

    try:
        # Fit models
        if category == '73292':
            params, _ = curve_fit(
                gompertz, current_time_index, category_data, maxfev=10000,
                bounds=(0, np.inf)
            )
            fitted_values = gompertz(current_time_index, *params)
            future_predicted = gompertz(future_time_index, *params)
        else:
            params, _ = curve_fit(
                bass_diffusion, current_time_index, category_data, maxfev=10000,
                bounds=(0, [1, 1, np.inf])
            )
            fitted_values = bass_diffusion(current_time_index, *params)
            future_predicted = bass_diffusion(future_time_index, *params)

        # Save predictions
        model_predictions[category] = future_predicted

        # Calculate confidence intervals
        ci_lower, ci_upper = calculate_bootstrap_ci(future_predicted)
        confidence_intervals[category] = (ci_lower, ci_upper)

    except Exception as e:
        print(f"Error fitting model for '{category}': {e}")
        continue

# Aggregate group predictions
group_1_predictions = np.sum([model_predictions.get(item, np.zeros(120)) for item in group_1_items], axis=0)
group_2_predictions = np.sum([model_predictions.get(item, np.zeros(120)) for item in group_2_items], axis=0)

# Aggregate group CIs
group_1_ci_lower = np.sum([confidence_intervals.get(item, (np.zeros(120),))[0] for item in group_1_items], axis=0)
group_1_ci_upper = np.sum([confidence_intervals.get(item, (np.zeros(120),))[1] for item in group_1_items], axis=0)

group_2_ci_lower = np.sum([confidence_intervals.get(item, (np.zeros(120),))[0] for item in group_2_items], axis=0)
group_2_ci_upper = np.sum([confidence_intervals.get(item, (np.zeros(120),))[1] for item in group_2_items], axis=0)

# Save group predictions and CIs
model_predictions['Group 1'] = group_1_predictions
confidence_intervals['Group 1'] = (group_1_ci_lower, group_1_ci_upper)

model_predictions['Group 2'] = group_2_predictions
confidence_intervals['Group 2'] = (group_2_ci_lower, group_2_ci_upper)

# Create future index
future_index = pd.date_range(
    start=historical_data['73292'].index[-1] + pd.offsets.MonthEnd(1),
    periods=120,
    freq='M'
)

# Plotting
plt.figure(figsize=(14, 7))

# Plot 73292
if '73292' in model_predictions and '73292' in confidence_intervals:
    ci_lower, ci_upper = confidence_intervals['73292']
    plt.plot(historical_data['73292'].index, historical_data['73292'], label='73292 (Historical)', linestyle='-')
    plt.plot(future_index, model_predictions['73292'], label='73292 (Predicted)', linestyle='--')
    plt.fill_between(future_index, ci_lower, ci_upper, alpha=0.2, label='73292 (95% CI)')

# Plot Group 1
if 'Group 1' in model_predictions and 'Group 1' in confidence_intervals:
    ci_lower, ci_upper = confidence_intervals['Group 1']
    plt.plot(future_index, model_predictions['Group 1'], label='Group 1 (Predicted)', linestyle='--')
    plt.fill_between(future_index, ci_lower, ci_upper, alpha=0.2, label='Group 1 (95% CI)')

# Plot Group 2
if 'Group 2' in model_predictions and 'Group 2' in confidence_intervals:
    ci_lower, ci_upper = confidence_intervals['Group 2']
    plt.plot(future_index, model_predictions['Group 2'], label='Group 2 (Predicted)', linestyle='--')
    plt.fill_between(future_index, ci_lower, ci_upper, alpha=0.2, label='Group 2 (95% CI)')

# Finalize plot
plt.xlabel('Date')
plt.ylabel('Value')
plt.title('Historical and Predicted Values with Bootstrapped 95% Confidence Intervals')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from bootstrapped import bootstrap as bs
from bootstrapped import stats_functions as bs_stats

# Load the data
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Define the individual components of each group
group_1_items = [item for item in filtered_data['Item'].unique() if item not in ['73292', '73358', '73359']]
group_2_items = ['73358', '73359']

# Define model functions
def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

# Initialize containers
model_predictions = {}
confidence_intervals = {}
historical_data = {}

# Function to calculate bootstrapped CIs
def calculate_bootstrap_ci(data):
    ci = bs.bootstrap(data, stat_func=bs_stats.mean, alpha=0.05)
    return ci.lower_bound, ci.upper_bound

# Process each category
for category in ['73292'] + group_1_items + group_2_items:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Convert to NumPy array
    category_data = category_data.values

    # Create time indices
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + 120)

    try:
        # Fit models
        if category == '73292':
            params, _ = curve_fit(
                gompertz, current_time_index, category_data, maxfev=10000,
                bounds=(0, np.inf)
            )
            fitted_values = gompertz(current_time_index, *params)
            future_predicted = gompertz(future_time_index, *params)
        else:
            params, _ = curve_fit(
                bass_diffusion, current_time_index, category_data, maxfev=10000,
                bounds=(0, [1, 1, np.inf])
            )
            fitted_values = bass_diffusion(current_time_index, *params)
            future_predicted = bass_diffusion(future_time_index, *params)

        # Save predictions
        model_predictions[category] = future_predicted

        # Calculate confidence intervals
        ci_lower = []
        ci_upper = []
        for prediction in future_predicted:
            # Use a small range of values around each prediction for bootstrapping
            sample_data = np.random.normal(loc=prediction, scale=prediction * 0.1, size=1000)
            ci = calculate_bootstrap_ci(sample_data)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        confidence_intervals[category] = (np.array(ci_lower), np.array(ci_upper))

    except Exception as e:
        print(f"Error fitting model for '{category}': {e}")
        continue

# Aggregate group predictions
group_1_predictions = np.sum([model_predictions.get(item, np.zeros(120)) for item in group_1_items], axis=0)
group_2_predictions = np.sum([model_predictions.get(item, np.zeros(120)) for item in group_2_items], axis=0)

# Aggregate group CIs
group_1_ci_lower = np.sum([confidence_intervals.get(item, (np.zeros(120),))[0] for item in group_1_items], axis=0)
group_1_ci_upper = np.sum([confidence_intervals.get(item, (np.zeros(120),))[1] for item in group_1_items], axis=0)

group_2_ci_lower = np.sum([confidence_intervals.get(item, (np.zeros(120),))[0] for item in group_2_items], axis=0)
group_2_ci_upper = np.sum([confidence_intervals.get(item, (np.zeros(120),))[1] for item in group_2_items], axis=0)

# Save group predictions and CIs
model_predictions['Group 1'] = group_1_predictions
confidence_intervals['Group 1'] = (group_1_ci_lower, group_1_ci_upper)

model_predictions['Group 2'] = group_2_predictions
confidence_intervals['Group 2'] = (group_2_ci_lower, group_2_ci_upper)

# Create future index
future_index = pd.date_range(
    start=historical_data['73292'].index[-1] + pd.offsets.MonthEnd(1),
    periods=120,
    freq='M'
)

# Plotting
plt.figure(figsize=(14, 7))

# Plot 73292
if '73292' in model_predictions and '73292' in confidence_intervals:
    ci_lower, ci_upper = confidence_intervals['73292']
    plt.plot(historical_data['73292'].index, historical_data['73292'], label='73292 (Historical)', linestyle='-')
    plt.plot(future_index, model_predictions['73292'], label='73292 (Predicted)', linestyle='--')
    plt.fill_between(future_index, ci_lower, ci_upper, alpha=0.2, label='73292 (95% CI)')

# Plot Group 1
if 'Group 1' in model_predictions and 'Group 1' in confidence_intervals:
    ci_lower, ci_upper = confidence_intervals['Group 1']
    plt.plot(future_index, model_predictions['Group 1'], label='Group 1 (Predicted)', linestyle='--')
    plt.fill_between(future_index, ci_lower, ci_upper, alpha=0.2, label='Group 1 (95% CI)')

# Plot Group 2
if 'Group 2' in model_predictions and 'Group 2' in confidence_intervals:
    ci_lower, ci_upper = confidence_intervals['Group 2']
    plt.plot(future_index, model_predictions['Group 2'], label='Group 2 (Predicted)', linestyle='--')
    plt.fill_between(future_index, ci_lower, ci_upper, alpha=0.2, label='Group 2 (95% CI)')

# Finalize plot
plt.xlabel('Date')
plt.ylabel('Value')
# plt.title('Historical and Predicted Values with Bootstrapped 95% Confidence Intervals')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from bootstrapped import bootstrap as bs
from bootstrapped import stats_functions as bs_stats

# Load the data
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Define the individual components of each group
group_1_items = [item for item in filtered_data['Item'].unique() if item not in ['73292', '73358', '73359']]
group_2_items = ['73358', '73359']

# Define model functions
def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

# Initialize containers
model_predictions = {}
confidence_intervals = {}
historical_data = {}

# Function to calculate bootstrapped CIs
def calculate_bootstrap_ci(data):
    ci = bs.bootstrap(data, stat_func=bs_stats.mean, alpha=0.05)
    return ci.lower_bound, ci.upper_bound

# Process each category
for category in ['73292'] + group_1_items + group_2_items:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('ME').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Convert to NumPy array
    category_data = category_data.values

    # Create time indices
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + 120)

    try:
        # Fit models
        if category == '73292':
            params, _ = curve_fit(
                gompertz, current_time_index, category_data, maxfev=10000,
                bounds=(0, np.inf)
            )
            fitted_values = gompertz(current_time_index, *params)
            future_predicted = gompertz(future_time_index, *params)
        else:
            params, _ = curve_fit(
                bass_diffusion, current_time_index, category_data, maxfev=10000,
                bounds=(0, [1, 1, np.inf])
            )
            fitted_values = bass_diffusion(current_time_index, *params)
            future_predicted = bass_diffusion(future_time_index, *params)

        # Save predictions
        model_predictions[category] = future_predicted

        # Calculate confidence intervals
        ci_lower = []
        ci_upper = []
        for prediction in future_predicted:
            sample_data = np.random.normal(loc=prediction, scale=prediction * 0.1, size=1000)
            ci = calculate_bootstrap_ci(sample_data)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        confidence_intervals[category] = (np.array(ci_lower), np.array(ci_upper))

    except Exception as e:
        print(f"Error fitting model for '{category}': {e}")
        continue

# Aggregate group predictions
group_1_predictions = np.sum([model_predictions.get(item, np.zeros(120)) for item in group_1_items], axis=0)
group_2_predictions = np.sum([model_predictions.get(item, np.zeros(120)) for item in group_2_items], axis=0)

# Aggregate group CIs
group_1_ci_lower = np.sum([confidence_intervals.get(item, (np.zeros(120),))[0] for item in group_1_items], axis=0)
group_1_ci_upper = np.sum([confidence_intervals.get(item, (np.zeros(120),))[1] for item in group_1_items], axis=0)

group_2_ci_lower = np.sum([confidence_intervals.get(item, (np.zeros(120),))[0] for item in group_2_items], axis=0)
group_2_ci_upper = np.sum([confidence_intervals.get(item, (np.zeros(120),))[1] for item in group_2_items], axis=0)

# Save group predictions and CIs
model_predictions['Group 1'] = group_1_predictions
confidence_intervals['Group 1'] = (group_1_ci_lower, group_1_ci_upper)

model_predictions['Group 2'] = group_2_predictions
confidence_intervals['Group 2'] = (group_2_ci_lower, group_2_ci_upper)

# Annual Results Table
table_data = []

# Historical Results (Last 5 Years)
for year in range(pd.Timestamp.now().year - 5, pd.Timestamp.now().year):
    row = {'Year': year}
    for category in ['73292', 'Group 1', 'Group 2']:
        if category in historical_data:
            annual_sum = historical_data[category].resample('A').sum().get(f'{year}-12-31', None)
            row[category] = annual_sum
        else:
            row[category] = None
    table_data.append(row)

# Predicted Results
for i, year in enumerate(range(pd.Timestamp.now().year, pd.Timestamp.now().year + 10)):
    row = {'Year': year}
    for category in ['73292', 'Group 1', 'Group 2']:
        if category in model_predictions:
            prediction = np.sum(model_predictions[category][i * 12:(i + 1) * 12])
            ci_lower, ci_upper = confidence_intervals.get(category, (np.zeros(120), np.zeros(120)))
            ci_low = np.sum(ci_lower[i * 12:(i + 1) * 12])
            ci_up = np.sum(ci_upper[i * 12:(i + 1) * 12])
            row[category] = f"{prediction:.0f} ({ci_low:.0f}, {ci_up:.0f})"
        else:
            row[category] = None
    table_data.append(row)

# Create DataFrame
results_df = pd.DataFrame(table_data)
results_df.set_index('Year', inplace=True)

results_df


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sktime.forecasting.model_selection import temporal_train_test_split
from sktime.forecasting.arima import AutoARIMA
from sktime.forecasting.base import ForecastingHorizon

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Rename the "Item" column values explicitly including 73358 and 73359 in "Group 1" and defining "Group 2"
items_to_keep = ['73292']
group_2_items = ['73298', '73358', '73359', '73401', '73402', '73422']
filtered_data.loc[~filtered_data['Item'].isin(items_to_keep + ['73358', '73359']), 'Item'] = 'Group 1'
filtered_data.loc[filtered_data['Item'].isin(group_2_items), 'Item'] = 'Group 2'

# List of items to fit models to
categories = ['Group 1', 'Group 2', '73292']

# Forecasting and Predictions using AutoARIMA
model_predictions = {}
historical_data = {}

for category in categories:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('M').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Split data into training and testing sets
    y_train, y_test = temporal_train_test_split(category_data, test_size=12)

    # Define forecasting horizon
    fh = ForecastingHorizon(np.arange(1, 13), is_relative=True)

    # Train AutoARIMA model
    forecaster = AutoARIMA()
    forecaster.fit(y_train)

    # Predict future values
    y_pred = forecaster.predict(fh)
    model_predictions[category] = y_pred

    # Extend predictions to 2035
    fh_future = ForecastingHorizon(np.arange(13, (2035 - y_train.index[-1].year + 1) * 12), is_relative=True)
    future_predictions = forecaster.predict(fh_future)
    model_predictions[category] = future_predictions

# Visualize the predictions for all categories on the same plot
plt.figure(figsize=(14, 7))

# Plot historical data
for category in categories:
    plt.plot(historical_data[category].index, historical_data[category], label=f'{category} (Historical)', linestyle='-')

# Plot predicted data
for category in categories:
    future_index = pd.date_range(
        start=historical_data[category].index[-1] + pd.offsets.MonthEnd(1),
        periods=len(model_predictions[category]),
        freq='M'
    )
    plt.plot(future_index, model_predictions[category], label=f'{category} (Predicted)', linestyle='--')

plt.title('Prediction Comparison for Group 1, Group 2, and 73292')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from autots import AutoTS

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Rename the "Item" column values explicitly including 73358 and 73359 in "Group 1" and defining "Group 2"
items_to_keep = ['73292']
group_2_items = ['73298', '73358', '73359', '73401', '73402', '73422']
filtered_data.loc[~filtered_data['Item'].isin(items_to_keep + ['73358', '73359']), 'Item'] = 'Group 1'
filtered_data.loc[filtered_data['Item'].isin(group_2_items), 'Item'] = 'Group 2'

# List of items to fit models to
categories = ['Group 1', 'Group 2', '73292']

model_predictions = {}
historical_data = {}

for category in categories:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('M').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Prepare data for AutoTS
    category_data = category_data.reset_index()
    category_data.columns = ['datetime', 'value']

    # Initialize AutoTS model
    model = AutoTS(
        forecast_length=12,
        frequency='M',
        ensemble='simple',
        model_list='superfast',  # Use "fast" or "superfast" for faster processing
        verbose=1,
    )

    # Fit the model
    model = model.fit(category_data, date_col='datetime', value_col='value')

    # Predict future values
    prediction = model.predict()
    forecast = prediction.forecast

    # Extend predictions to 2035
    model = AutoTS(
        forecast_length=(2035 - historical_data[category].index[-1].year) * 12,
        frequency='M',
        ensemble='simple',
        model_list='superfast',
        verbose=1,
    )
    model = model.fit(category_data, date_col='datetime', value_col='value')
    future_forecast = model.predict().forecast
    model_predictions[category] = future_forecast

    print(f"Best model for {category}: {model.best_model}")

# Visualize the predictions for all categories on the same plot
plt.figure(figsize=(14, 7))

# Plot historical data
for category in categories:
    plt.plot(historical_data[category].index, historical_data[category], label=f'{category} (Historical)', linestyle='-')

# Plot predicted data
for category in categories:
    future_index = pd.date_range(
        start=historical_data[category].index[-1] + pd.offsets.MonthEnd(1),
        periods=len(model_predictions[category]),
        freq='M'
    )
    plt.plot(future_index, model_predictions[category].values.flatten(), label=f'{category} (Predicted)', linestyle='--')

plt.title('Prediction Comparison for Group 1, Group 2, and 73292')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from autots import AutoTS

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Rename the "Item" column values explicitly including 73358 and 73359 in "Group 1" and defining "Group 2"
items_to_keep = ['73292']
group_2_items = ['73298', '73358', '73359', '73401', '73402', '73422']
filtered_data.loc[~filtered_data['Item'].isin(items_to_keep + ['73358', '73359']), 'Item'] = 'Group 1'
filtered_data.loc[filtered_data['Item'].isin(group_2_items), 'Item'] = 'Group 2'

# List of items to fit models to
categories = ['Group 1', 'Group 2', '73292']

model_predictions = {}
historical_data = {}

for category in categories:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('M').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Prepare data for AutoTS
    category_data = category_data.reset_index()
    category_data.columns = ['datetime', 'value']

    # Dynamically calculate forecast length
    max_forecast_length = min(12, len(category_data) // 2)

    # Initialize AutoTS model
    model = AutoTS(
        forecast_length=max_forecast_length,
        frequency='M',
        ensemble='simple',
        model_list='superfast',  # Use "fast" or "superfast" for faster processing
        verbose=1,
    )

    # Fit the model
    model = model.fit(category_data, date_col='datetime', value_col='value')

    # Predict future values
    prediction = model.predict()
    forecast = prediction.forecast

    # Extend predictions to 2035 using a feasible forecast length
    remaining_months = (2035 - historical_data[category].index[-1].year) * 12
    extended_forecast_length = min(remaining_months, len(category_data))

    model = AutoTS(
        forecast_length=extended_forecast_length,
        frequency='M',
        ensemble='simple',
        model_list='superfast',
        verbose=1,
    )
    model = model.fit(category_data, date_col='datetime', value_col='value')
    future_forecast = model.predict().forecast
    model_predictions[category] = future_forecast

    print(f"Best model for {category}: {model.best_model}")

# Visualize the predictions for all categories on the same plot
plt.figure(figsize=(14, 7))

# Plot historical data
for category in categories:
    plt.plot(historical_data[category].index, historical_data[category], label=f'{category} (Historical)', linestyle='-')

# Plot predicted data
for category in categories:
    future_index = pd.date_range(
        start=historical_data[category].index[-1] + pd.offsets.MonthEnd(1),
        periods=len(model_predictions[category]),
        freq='M'
    )
    plt.plot(future_index, model_predictions[category].values.flatten(), label=f'{category} (Predicted)', linestyle='--')

plt.title('Prediction Comparison for Group 1, Group 2, and 73292')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from autots import AutoTS

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Rename the "Item" column values explicitly including 73358 and 73359 in "Group 1" and defining "Group 2"
items_to_keep = ['73292']
group_2_items = ['73298', '73358', '73359', '73401', '73402', '73422']
filtered_data.loc[~filtered_data['Item'].isin(items_to_keep + ['73358', '73359']), 'Item'] = 'Group 1'
filtered_data.loc[filtered_data['Item'].isin(group_2_items), 'Item'] = 'Group 2'

# List of items to fit models to
categories = ['Group 1', 'Group 2', '73292']

model_predictions = {}
historical_data = {}

for category in categories:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('M').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Prepare data for AutoTS
    category_data = category_data.reset_index()
    category_data.columns = ['datetime', 'value']

    # Ensure sufficient data for training and forecasting
    if len(category_data) < 24:  # Minimum data length required for AutoTS
        print(f"Not enough data for '{category}'. Skipping...")
        continue

    # Dynamically calculate forecast length
    max_forecast_length = min(12, len(category_data) // 2)

    # Initialize AutoTS model
    model = AutoTS(
        forecast_length=max_forecast_length,
        frequency='M',
        ensemble='simple',
        model_list='superfast',  # Use "fast" or "superfast" for faster processing
        verbose=1,
    )

    # Fit the model
    try:
        model = model.fit(category_data, date_col='datetime', value_col='value')
        prediction = model.predict()
        forecast = prediction.forecast

        # Extend predictions to 2035 using feasible forecast length
        remaining_months = (2035 - historical_data[category].index[-1].year) * 12
        extended_forecast_length = min(remaining_months, len(category_data))

        model = AutoTS(
            forecast_length=extended_forecast_length,
            frequency='M',
            ensemble='simple',
            model_list='superfast',
            verbose=1,
        )
        model = model.fit(category_data, date_col='datetime', value_col='value')
        future_forecast = model.predict().forecast
        model_predictions[category] = future_forecast

        print(f"Best model for {category}: {model.best_model}")
    except Exception as e:
        print(f"Error fitting model for '{category}': {e}")
        continue

# Visualize the predictions for all categories on the same plot
plt.figure(figsize=(14, 7))

# Plot historical data
for category in categories:
    if category in historical_data:
        plt.plot(historical_data[category].index, historical_data[category], label=f'{category} (Historical)', linestyle='-')

# Plot predicted data
for category in categories:
    if category in model_predictions:
        future_index = pd.date_range(
            start=historical_data[category].index[-1] + pd.offsets.MonthEnd(1),
            periods=len(model_predictions[category]),
            freq='M'
        )
        plt.plot(future_index, model_predictions[category].values.flatten(), label=f'{category} (Predicted)', linestyle='--')

plt.title('Prediction Comparison for Group 1, Group 2, and 73292')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from autots import AutoTS

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Rename the "Item" column values explicitly including 73358 and 73359 in "Group 1" and defining "Group 2"
items_to_keep = ['73292']
group_2_items = ['73298', '73358', '73359', '73401', '73402', '73422']
filtered_data.loc[~filtered_data['Item'].isin(items_to_keep + ['73358', '73359']), 'Item'] = 'Group 1'
filtered_data.loc[filtered_data['Item'].isin(group_2_items), 'Item'] = 'Group 2'

# List of items to fit models to
categories = ['Group 1', 'Group 2', '73292']

model_predictions = {}
historical_data = {}

for category in categories:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('M').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Prepare data for AutoTS
    category_data = category_data.reset_index()
    category_data.columns = ['datetime', 'value']

    # Ensure sufficient data for training and forecasting
    if len(category_data) < 24:  # Minimum data length required for AutoTS
        print(f"Not enough data for '{category}'. Skipping...")
        continue

    # Dynamically calculate forecast length
    max_forecast_length = min(12, len(category_data) // 2)

    # Initialize AutoTS model
    model = AutoTS(
        forecast_length=max_forecast_length,
        frequency='infer',  # Automatically infer frequency from data
        ensemble='simple',
        model_list='fast',  # Use "fast" or "superfast" for faster processing
        verbose=1,
    )

    # Fit the model
    try:
        model = model.fit(category_data, date_col='datetime', value_col='value')
        prediction = model.predict()
        forecast = prediction.forecast

        # Extend predictions to 2035 using feasible forecast length
        remaining_months = (2035 - historical_data[category].index[-1].year) * 12
        extended_forecast_length = min(remaining_months, max_forecast_length)

        model = AutoTS(
            forecast_length=extended_forecast_length,
            frequency='infer',
            ensemble='simple',
            model_list='fast',
            verbose=1,
        )
        model = model.fit(category_data, date_col='datetime', value_col='value')
        future_forecast = model.predict().forecast
        model_predictions[category] = future_forecast

        print(f"Best model for {category}: {model.best_model}")
    except Exception as e:
        print(f"Error fitting model for '{category}': {e}")
        continue

# Visualize the predictions for all categories on the same plot
plt.figure(figsize=(14, 7))

# Plot historical data
for category in categories:
    if category in historical_data:
        plt.plot(historical_data[category].index, historical_data[category], label=f'{category} (Historical)', linestyle='-')

# Plot predicted data
for category in categories:
    if category in model_predictions:
        future_index = pd.date_range(
            start=historical_data[category].index[-1] + pd.offsets.MonthEnd(1),
            periods=len(model_predictions[category]),
            freq='M'
        )
        plt.plot(future_index, model_predictions[category].values.flatten(), label=f'{category} (Predicted)', linestyle='--')

plt.title('Prediction Comparison for Group 1, Group 2, and 73292')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from autots import AutoTS

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Rename the "Item" column values explicitly including 73358 and 73359 in "Group 1" and defining "Group 2"
items_to_keep = ['73292']
group_2_items = ['73298', '73358', '73359', '73401', '73402', '73422']
filtered_data.loc[~filtered_data['Item'].isin(items_to_keep + ['73358', '73359']), 'Item'] = 'Group 1'
filtered_data.loc[filtered_data['Item'].isin(group_2_items), 'Item'] = 'Group 2'

# List of items to fit models to
categories = ['Group 1', 'Group 2', '73292']

model_predictions = {}
historical_data = {}

for category in categories:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('M').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Prepare data for AutoTS
    category_data = category_data.reset_index()
    category_data.columns = ['datetime', 'value']

    # Ensure sufficient data for training and forecasting
    if len(category_data) < 24:  # Minimum data length required for AutoTS
        print(f"Not enough data for '{category}'. Skipping...")
        continue

    # Dynamically calculate forecast length
    forecast_length = 120  # Extend predictions for 10 years (120 months)

    # Initialize AutoTS model
    model = AutoTS(
        forecast_length=forecast_length,
        frequency='infer',  # Automatically infer frequency from data
        ensemble='simple',
        model_list='fast',  # Use "fast" or "superfast" for faster processing
        verbose=1,
    )

    # Fit the model
    try:
        model = model.fit(category_data, date_col='datetime', value_col='value')
        prediction = model.predict()
        future_forecast = prediction.forecast
        model_predictions[category] = future_forecast

        print(f"Best model for {category}: {model.best_model}")
    except Exception as e:
        print(f"Error fitting model for '{category}': {e}")
        continue

# Visualize the predictions for all categories on the same plot
plt.figure(figsize=(14, 7))

# Plot historical data
for category in categories:
    if category in historical_data:
        plt.plot(historical_data[category].index, historical_data[category], label=f'{category} (Historical)', linestyle='-')

# Plot predicted data with 5-year and 10-year markers
for category in categories:
    if category in model_predictions:
        future_index = pd.date_range(
            start=historical_data[category].index[-1] + pd.offsets.MonthEnd(1),
            periods=len(model_predictions[category]),
            freq='M'
        )
        plt.plot(future_index, model_predictions[category].values.flatten(), label=f'{category} (Predicted)', linestyle='--')

        # Highlight 5-year and 10-year points
        five_year_index = future_index[60] if len(future_index) > 60 else None
        ten_year_index = future_index[119] if len(future_index) > 119 else None

        if five_year_index is not None:
            plt.axvline(five_year_index, color='blue', linestyle=':', label=f'{category} 5-Year Marker')
        if ten_year_index is not None:
            plt.axvline(ten_year_index, color='green', linestyle=':', label=f'{category} 10-Year Marker')

plt.title('10-Year Prediction Comparison with Markers')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from autots import AutoTS

# Load the data (use your own path as needed)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Rename the "Item" column values explicitly
items_to_keep = ['73292']
group_2_items = ['73298', '73358', '73359', '73401', '73402', '73422']
filtered_data.loc[~filtered_data['Item'].isin(items_to_keep + ['73358', '73359']), 'Item'] = 'Group 1'
filtered_data.loc[filtered_data['Item'].isin(group_2_items), 'Item'] = 'Group 2'

# List of items to fit models to
categories = ['Group 1', 'Group 2', '73292']

model_predictions = {}
historical_data = {}

for category in categories:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('M').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Prepare data for AutoTS
    category_data = category_data.reset_index()
    category_data.columns = ['datetime', 'value']

    # Dynamically determine maximum forecast length for cross-validation
    max_forecast_length = len(category_data) // 2  # Use half the available data for training
    if max_forecast_length < 1:
        print(f"Not enough data for '{category}'. Skipping...")
        continue

    # Initialize AutoTS model for cross-validation
    model = AutoTS(
        forecast_length=max_forecast_length,
        frequency='infer',
        ensemble='simple',
        model_list='fast',
        verbose=1,
    )

    # Fit the model and select the best
    try:
        model = model.fit(category_data, date_col='datetime', value_col='value')
        print(f"Best model for {category}: {model.best_model}")
        
        # Use the selected model to forecast for 10 years
        forecast_length = 120  # 10 years
        prediction = model.predict(forecast_length=forecast_length)
        future_forecast = prediction.forecast
        model_predictions[category] = future_forecast

    except Exception as e:
        print(f"Error fitting model for '{category}': {e}")
        continue

# Visualize the predictions for all categories on the same plot
plt.figure(figsize=(14, 7))

# Plot historical data
for category in categories:
    if category in historical_data:
        plt.plot(historical_data[category].index, historical_data[category], label=f'{category} (Historical)', linestyle='-')

# Plot predicted data with 5-year and 10-year markers
for category in categories:
    if category in model_predictions:
        future_index = pd.date_range(
            start=historical_data[category].index[-1] + pd.offsets.MonthEnd(1),
            periods=len(model_predictions[category]),
            freq='M'
        )
        plt.plot(future_index, model_predictions[category].values.flatten(), label=f'{category} (Predicted)', linestyle='--')

        # Highlight 5-year and 10-year points
        five_year_index = future_index[60] if len(future_index) > 60 else None
        ten_year_index = future_index[119] if len(future_index) > 119 else None

        if five_year_index is not None:
            plt.axvline(five_year_index, color='blue', linestyle=':', label=f'{category} 5-Year Marker')
        if ten_year_index is not None:
            plt.axvline(ten_year_index, color='green', linestyle=':', label=f'{category} 10-Year Marker')

plt.title('10-Year Prediction Comparison with Markers')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from bootstrapped import bootstrap as bs
from bootstrapped import stats_functions as bs_stats

# Load the data
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Define the individual components of each group
group_1_items = [item for item in filtered_data['Item'].unique() if item not in ['73292', '73358', '73359']]
group_2_items = ['73358', '73359']

# Define model functions
def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

# Initialize containers
model_predictions = {}
confidence_intervals = {}
historical_data = {}

# Function to calculate bootstrapped CIs
def calculate_bootstrap_ci(data):
    ci = bs.bootstrap(data, stat_func=bs_stats.mean, alpha=0.05)
    return ci.lower_bound, ci.upper_bound

# Process each category
for category in ['73292'] + group_1_items + group_2_items:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('Y').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Save historical data
    historical_data[category] = category_data

    # Convert to NumPy array
    category_data = category_data.values

    # Create time indices
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + 10)  # Predict for 10 years

    try:
        # Fit models
        if category == '73292':
            params, _ = curve_fit(
                gompertz, current_time_index, category_data, maxfev=10000,
                bounds=(0, np.inf)
            )
            fitted_values = gompertz(current_time_index, *params)
            future_predicted = gompertz(future_time_index, *params)
        else:
            params, _ = curve_fit(
                bass_diffusion, current_time_index, category_data, maxfev=10000,
                bounds=(0, [1, 1, np.inf])
            )
            fitted_values = bass_diffusion(current_time_index, *params)
            future_predicted = bass_diffusion(future_time_index, *params)

        # Save predictions
        model_predictions[category] = future_predicted

        # Calculate confidence intervals
        ci_lower = []
        ci_upper = []
        for prediction in future_predicted:
            # Use a small range of values around each prediction for bootstrapping
            sample_data = np.random.normal(loc=prediction, scale=prediction * 0.1, size=1000)
            ci = calculate_bootstrap_ci(sample_data)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        confidence_intervals[category] = (np.array(ci_lower), np.array(ci_upper))

    except Exception as e:
        print(f"Error fitting model for '{category}': {e}")
        continue

# Aggregate group predictions
group_1_predictions = np.sum([model_predictions.get(item, np.zeros(10)) for item in group_1_items], axis=0)
group_2_predictions = np.sum([model_predictions.get(item, np.zeros(10)) for item in group_2_items], axis=0)

# Aggregate group CIs
group_1_ci_lower = np.sum([confidence_intervals.get(item, (np.zeros(10),))[0] for item in group_1_items], axis=0)
group_1_ci_upper = np.sum([confidence_intervals.get(item, (np.zeros(10),))[1] for item in group_1_items], axis=0)

group_2_ci_lower = np.sum([confidence_intervals.get(item, (np.zeros(10),))[0] for item in group_2_items], axis=0)
group_2_ci_upper = np.sum([confidence_intervals.get(item, (np.zeros(10),))[1] for item in group_2_items], axis=0)

# Save group predictions and CIs
model_predictions['Group 1'] = group_1_predictions
confidence_intervals['Group 1'] = (group_1_ci_lower, group_1_ci_upper)

model_predictions['Group 2'] = group_2_predictions
confidence_intervals['Group 2'] = (group_2_ci_lower, group_2_ci_upper)

# Create the results table
results_table = []

# Add last five years of historical data
for category in ['73292', 'Group 1', 'Group 2']:
    if category in historical_data:
        hist_data = historical_data[category].iloc[-5:]
        for year, value in hist_data.items():
            results_table.append({
                'Year': year.year,
                'Category': category,
                'Value': value,
                'Lower CI': np.nan,
                'Upper CI': np.nan
            })

# Add predictions and CIs
for i, year in enumerate(range(2025, 2025 + 10)):
    for category in ['73292', 'Group 1', 'Group 2']:
        if category in model_predictions:
            results_table.append({
                'Year': year,
                'Category': category,
                'Value': model_predictions[category][i],
                'Lower CI': confidence_intervals[category][0][i],
                'Upper CI': confidence_intervals[category][1][i]
            })

# Convert to DataFrame
results_df = pd.DataFrame(results_table)


In [ ]:
results_df

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from bootstrapped import bootstrap as bs
from bootstrapped import stats_functions as bs_stats

# Load the data
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/MBSGeneticsBenefit_202410.feather"
data = pd.read_feather(file_path)

# Strip whitespace from the "Item" column
data['Item'] = data['Item'].astype(str).str.strip()

# Ensure the "Month" column is treated as datetime and set as index
data['Month'] = pd.to_datetime(data['Month']).dt.to_period('M').dt.to_timestamp()
data.set_index('Month', inplace=True)

# Filter the data to include only specific "Item" categories
items_to_include = [
    '73292', '73298', '73299', '73358', '73359',
    '73360', '73395', '73401', '73402', '73422',
    '73425', '73426', '73442', '73456', '73457', '73458'
]
filtered_data = data[data['Item'].isin(items_to_include)].copy()

# Define the individual components of each group
group_1_items = [item for item in filtered_data['Item'].unique() if item not in ['73292', '73358', '73359']]
group_2_items = ['73358', '73359']

# Define model functions
def gompertz(t, a, b, c):
    return a * np.exp(-b * np.exp(-c * t))

def bass_diffusion(t, p, q, M):
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

# Initialize containers
model_predictions = {}
confidence_intervals = {}
historical_data = {}

# Function to calculate bootstrapped CIs
def calculate_bootstrap_ci(data):
    ci = bs.bootstrap(data, stat_func=bs_stats.mean, alpha=0.05)
    return ci.lower_bound, ci.upper_bound

# Process each category
for category in ['73292'] + group_1_items + group_2_items:
    category_data = filtered_data[filtered_data['Item'] == category]['Value'].resample('Y').sum().dropna()

    if category_data.empty:
        print(f"No data found for '{category}'. Skipping...")
        continue

    # Convert index to DatetimeIndex
    if not isinstance(category_data.index, pd.DatetimeIndex):
        category_data.index = pd.to_datetime(category_data.index)

    # Save historical data
    historical_data[category] = category_data

    # Convert to NumPy array
    category_data = category_data.values

    # Create time indices
    current_time_index = np.arange(len(category_data))
    future_time_index = np.arange(len(category_data), len(category_data) + 10)  # Predict for 10 years

    try:
        # Fit models
        if category == '73292':
            params, _ = curve_fit(
                gompertz, current_time_index, category_data, maxfev=10000,
                bounds=(0, np.inf)
            )
            fitted_values = gompertz(current_time_index, *params)
            future_predicted = gompertz(future_time_index, *params)
        else:
            params, _ = curve_fit(
                bass_diffusion, current_time_index, category_data, maxfev=10000,
                bounds=(0, [1, 1, np.inf])
            )
            fitted_values = bass_diffusion(current_time_index, *params)
            future_predicted = bass_diffusion(future_time_index, *params)

        # Save predictions
        model_predictions[category] = future_predicted

        # Calculate confidence intervals
        ci_lower = []
        ci_upper = []
        for prediction in future_predicted:
            # Use a small range of values around each prediction for bootstrapping
            sample_data = np.random.normal(loc=prediction, scale=prediction * 0.1, size=1000)
            ci = calculate_bootstrap_ci(sample_data)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        confidence_intervals[category] = (np.array(ci_lower), np.array(ci_upper))

    except Exception as e:
        print(f"Error fitting model for '{category}': {e}")
        continue

# Aggregate group predictions
group_1_predictions = np.sum([model_predictions.get(item, np.zeros(10)) for item in group_1_items], axis=0)
group_2_predictions = np.sum([model_predictions.get(item, np.zeros(10)) for item in group_2_items], axis=0)

# Aggregate group CIs
group_1_ci_lower = np.sum([confidence_intervals.get(item, (np.zeros(10),))[0] for item in group_1_items], axis=0)
group_1_ci_upper = np.sum([confidence_intervals.get(item, (np.zeros(10),))[1] for item in group_1_items], axis=0)

group_2_ci_lower = np.sum([confidence_intervals.get(item, (np.zeros(10),))[0] for item in group_2_items], axis=0)
group_2_ci_upper = np.sum([confidence_intervals.get(item, (np.zeros(10),))[1] for item in group_2_items], axis=0)

# Save group predictions and CIs
model_predictions['Group 1'] = group_1_predictions
confidence_intervals['Group 1'] = (group_1_ci_lower, group_1_ci_upper)

model_predictions['Group 2'] = group_2_predictions
confidence_intervals['Group 2'] = (group_2_ci_lower, group_2_ci_upper)

# Create the results table
years = list(historical_data['73292'].index.year[-5:]) + list(range(2025, 2025 + 10))
results = []

for year_index, year in enumerate(years):
    row = {'Year': year}
    for category in ['73292', 'Group 1', 'Group 2']:
        hist_data = historical_data.get(category, pd.Series())
        if not hist_data.empty and year in hist_data.index.year:
            value = int(hist_data[hist_data.index.year == year].values[0])
            ci_text = "(N/A)"
        else:
            value = int(model_predictions[category][year_index - 5])
            ci_lower = int(confidence_intervals[category][0][year_index - 5])
            ci_upper = int(confidence_intervals[category][1][year_index - 5])
            ci_text = f"({ci_lower}–{ci_upper})"
        row[category] = f"{value} {ci_text}"
    results.append(row)

# Convert to DataFrame
results_df = pd.DataFrame(results)


In [ ]:
results_df

In [ ]:
## Benefit